<!-- NOTEBOOK_OVERVIEW -->
# 1. Hybrid Routing and Fusion Evaluation

## 2. Introduction
This notebook evaluates the semantic-anchored hybrid model family. It compares fixed routing, IBVS-conditioned routing, veto-style routing, and learned fusion strategies under a strict ID-only tuning policy and then aggregates hybrid results into the main repeatability/reporting tables.

## 3. Workflow Steps
1. Load the split-specific processed dataset and rebuild the semantic, lexical, flag, and IBVS feature pathways needed by the hybrid variants.
2. Train or reconstruct the component models (`semantic`, `M1`, `M3`) required by the routing strategies.
3. Tune hybrid parameters on `VAL` only using the notebook’s ID-only objective and feasibility constraints.
4. Evaluate each hybrid variant on `VAL`, `TEST`, and all configured OOD sets.
5. Write canonical hybrid metrics and rebuild the repeatability and difficulty-bin tables from the per-family outputs.

## 4. Evaluation and Protocol Notes
1. All hybrid parameter search is `id_val_only`; OOD sets are never used for tuning.
2. Reported hybrid metrics include:
   - `accuracy`, `macro_f1`
   - `ROC-AUC`, `AUC-PR` where applicable through the shared evaluation outputs
   - `TPR@1% FPR`, `TPR@5% FPR`, `TPR@10% FPR`
   - routing behavior such as `semantic_coverage` and `defer_rate`
3. The notebook evaluates multiple hybrid mechanisms, including baseline fallback routing, anchored boost/veto variants, expert gating, and learned meta fusion.
4. This notebook also rebuilds:
   - `results_table_v2_repeatability.csv`
   - `results_table_difficulty_bins.csv`
   from the split-level metrics written by notebooks 04, 05, and 06.

## 5. Execution Notes
1. Set `SPLIT_TAG` (`A`, `B`, `C`) before execution.
2. Use `WRITE_MINIMAL_OUTPUTS=0` to regenerate secondary OOD artifacts and suffixed diagnostics.
3. For complete repeatability regeneration, ensure notebooks 04 and 05 have already written the current split-level canonical metrics.


In [1]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 1) Imports + Configs

from pathlib import Path
import sys
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sentence_transformers import SentenceTransformer

from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [2]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 2) Load processed dataset v2

import os

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Reload local modules to avoid stale notebook-kernel imports.
import importlib
import src.evaluation.eval_metrics as eval_metrics
import src.features.ibvs as ibvs_mod

importlib.reload(eval_metrics)
importlib.reload(ibvs_mod)

from src.evaluation.eval_metrics import (
    evaluate_predictions,
    results_to_dataframe,
    best_threshold_by_macro_f1,
    best_threshold_low_fpr_with_macro_guard,
)

from src.features.ibvs import ibvs_v2_with_triggers, IBVS_V2_NUMERIC_COLUMNS
from src.common.notebook_utils import (
    DEFAULT_OVERRIDE_PATTERNS,
    encode_texts,
    get_git_commit,
    make_flag_matrix,
    safe_qcut,
    text_stats,
)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# ============================
# DATASET SPLIT SELECTION
# Split A = original v2 (current baseline)
# Split B/C = repeatability resamples (same rows, new ID train/val/test, OOD fixed)
# ============================

SPLIT_TAG = os.getenv("SPLIT_TAG", "B").strip().upper()   # env override: A/B/C
WRITE_MINIMAL_OUTPUTS = os.getenv("WRITE_MINIMAL_OUTPUTS", "1").strip().lower() not in {"0", "false", "no"}
print("WRITE_MINIMAL_OUTPUTS:", WRITE_MINIMAL_OUTPUTS)

expected_filename_by_split = {
    "A": "jailbreak_benchmarks_processed_v2.csv",
    "B": "jailbreak_benchmarks_processed_v2_splitB.csv",
    "C": "jailbreak_benchmarks_processed_v2_splitC.csv",
}
if SPLIT_TAG not in expected_filename_by_split:
    raise ValueError("SPLIT_TAG must be 'A', 'B', or 'C'.")
processed_filename = expected_filename_by_split[SPLIT_TAG]

processed_path = DATA_PROCESSED / processed_filename
assert processed_path.name == expected_filename_by_split[SPLIT_TAG], "processed filename/split mismatch"
print("Using dataset:", processed_path)

df = pd.read_csv(processed_path)

split_counts = df["split"].value_counts()
required_splits = ["train", "val", "test", "ood_test"]
missing_splits = [s for s in required_splits if s not in split_counts.index]
if missing_splits:
    raise ValueError(f"Missing required split(s): {missing_splits}")

print("Rows:", len(df))
print("\nSplit counts:")
print(split_counts)
print("\nLabel counts:")
print(df["label"].value_counts())


GIT_COMMIT = get_git_commit(PROJECT_ROOT)
print("Git commit:", GIT_COMMIT)


WRITE_MINIMAL_OUTPUTS: False
Using dataset: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/data/processed/jailbreak_benchmarks_processed_v2_splitC.csv
Rows: 6424

Split counts:
split
ood_test_injection_standard    3986
ood_test                        768
train                           694
ood_test_injection              678
test                            149
val                             149
Name: count, dtype: int64

Label counts:
label
1    3437
0    2987
Name: count, dtype: int64
Git commit: 221e1583bedb737ba71fad54d220c5f495f619e8


In [3]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# Split views

df_train = df[df["split"] == "train"].copy()
df_val   = df[df["split"] == "val"].copy()
df_test  = df[df["split"] == "test"].copy()

OOD_SPLIT_ORDER = ["ood_test", "ood_test_injection", "ood_test_injection_standard"]
df_ood_map = {}
for split_name in OOD_SPLIT_ORDER:
    d = df[df["split"] == split_name].copy()
    if not d.empty:
        df_ood_map[split_name] = d

if "ood_test" not in df_ood_map:
    raise ValueError("Missing required split 'ood_test'.")

df_ood = df_ood_map["ood_test"]
df_ood_injection = df_ood_map.get("ood_test_injection")

for name, d in [("train", df_train), ("val", df_val), ("test", df_test)] + list(df_ood_map.items()):
    if d.empty:
        raise ValueError(f"Split '{name}' is empty.")
    print(f"{name:18s}", d.shape, d["label"].value_counts().to_dict())

y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values
y_ood   = df_ood["label"].values
y_ood_map = {k: v["label"].values for k, v in df_ood_map.items()}
y_ood_injection = y_ood_map.get("ood_test_injection")

OOD_SECONDARY_SPLITS = [k for k in OOD_SPLIT_ORDER if k in df_ood_map and k != "ood_test"]


train              (694, 16) {1: 504, 0: 190}
val                (149, 16) {1: 109, 0: 40}
test               (149, 16) {1: 108, 0: 41}
ood_test           (768, 16) {0: 384, 1: 384}
ood_test_injection (678, 16) {0: 339, 1: 339}
ood_test_injection_standard (3986, 16) {1: 1993, 0: 1993}


In [4]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 3) Build fallback features: TF-IDF + FLAGS + IBVS v2 structured (M3)

# TF-IDF

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
)

tfidf.fit(df_train["prompt_text"].astype(str))

X_lex_train = tfidf.transform(df_train["prompt_text"].astype(str))
X_lex_val   = tfidf.transform(df_val["prompt_text"].astype(str))
X_lex_test  = tfidf.transform(df_test["prompt_text"].astype(str))

X_lex_ood_map = {
    split_name: tfidf.transform(df_ood_map[split_name]["prompt_text"].astype(str))
    for split_name in df_ood_map
}
X_lex_ood = X_lex_ood_map["ood_test"]
X_lex_ood_injection = X_lex_ood_map.get("ood_test_injection")

print("TF-IDF shapes:", X_lex_train.shape, X_lex_val.shape, X_lex_test.shape, X_lex_ood.shape)
for split_name, X in X_lex_ood_map.items():
    print(f"{split_name:18s}", X.shape)


TF-IDF shapes: (694, 2290) (149, 2290) (149, 2290) (768, 2290)
ood_test           (768, 2290)
ood_test_injection (678, 2290)
ood_test_injection_standard (3986, 2290)


In [5]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# FLAGS + IBVS v2 structured features
OVERRIDE_PATTERNS = DEFAULT_OVERRIDE_PATTERNS

def make_ibvs_v2_struct_matrix(df_split: pd.DataFrame):
    rows = []
    totals = []
    for t in df_split["prompt_text"].astype(str):
        total, breakdown, _ = ibvs_v2_with_triggers(t)
        rows.append(breakdown)
        totals.append(total)

    comp = pd.DataFrame(rows)
    for c in IBVS_V2_NUMERIC_COLUMNS:
        if c not in comp.columns:
            comp[c] = 0.0
    comp = comp[list(IBVS_V2_NUMERIC_COLUMNS)].astype(float)
    comp["ibvs_v2_total"] = np.array(totals, dtype=float)
    return csr_matrix(comp.values.astype(float)), comp


In [6]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# Purpose: 06_hybrid_routing.ipynb
X_flag_train = make_flag_matrix(df_train)
X_flag_val   = make_flag_matrix(df_val)
X_flag_test  = make_flag_matrix(df_test)

X_flag_ood_map = {split_name: make_flag_matrix(df_ood_map[split_name]) for split_name in df_ood_map}
X_flag_ood = X_flag_ood_map["ood_test"]

X_ibvs_train, ibvs_train_df = make_ibvs_v2_struct_matrix(df_train)
X_ibvs_val, ibvs_val_df     = make_ibvs_v2_struct_matrix(df_val)
X_ibvs_test, ibvs_test_df   = make_ibvs_v2_struct_matrix(df_test)

X_ibvs_ood_map = {}
ibvs_ood_df_map = {}
for split_name, d in df_ood_map.items():
    X_ibvs_o, ibvs_o_df = make_ibvs_v2_struct_matrix(d)
    X_ibvs_ood_map[split_name] = X_ibvs_o
    ibvs_ood_df_map[split_name] = ibvs_o_df

X_ibvs_ood = X_ibvs_ood_map["ood_test"]
ibvs_ood_df = ibvs_ood_df_map["ood_test"]

# M2 features (kept for comparison/debug)
X_m2_train = hstack([X_lex_train, X_flag_train]).tocsr()
X_m2_val   = hstack([X_lex_val,   X_flag_val]).tocsr()
X_m2_test  = hstack([X_lex_test,  X_flag_test]).tocsr()
X_m2_ood_map = {
    split_name: hstack([X_lex_ood_map[split_name], X_flag_ood_map[split_name]]).tocsr()
    for split_name in df_ood_map
}
X_m2_ood = X_m2_ood_map["ood_test"]

# M3 fallback features: TF-IDF + flags + IBVS v2 structured components
X_m3_train = hstack([X_lex_train, X_flag_train, X_ibvs_train]).tocsr()
X_m3_val   = hstack([X_lex_val,   X_flag_val,   X_ibvs_val]).tocsr()
X_m3_test  = hstack([X_lex_test,  X_flag_test,  X_ibvs_test]).tocsr()
X_m3_ood_map = {
    split_name: hstack([X_lex_ood_map[split_name], X_flag_ood_map[split_name], X_ibvs_ood_map[split_name]]).tocsr()
    for split_name in df_ood_map
}
X_m3_ood   = X_m3_ood_map["ood_test"]

print("M2 feature shapes:", X_m2_train.shape, X_m2_val.shape, X_m2_test.shape, X_m2_ood.shape)
print("M3 feature shapes:", X_m3_train.shape, X_m3_val.shape, X_m3_test.shape, X_m3_ood.shape)
for ood_name in OOD_SECONDARY_SPLITS:
    print(f"M3 secondary OOD feature shape ({ood_name}):", X_m3_ood_map[ood_name].shape)


M2 feature shapes: (694, 2298) (149, 2298) (149, 2298) (768, 2298)
M3 feature shapes: (694, 2317) (149, 2317) (149, 2317) (768, 2317)
M3 secondary OOD feature shape (ood_test_injection): (678, 2317)
M3 secondary OOD feature shape (ood_test_injection_standard): (3986, 2317)


In [7]:
# Cell Purpose: Train model(s) using the prepared feature sets and split configuration.
# 4) Train lexical M1 and fallback M3 models (XGBoost) + choose thresholds on VAL
# Threshold policy prioritises low-FPR operation while guarding macro-F1.

M3_TARGET_FPR = 0.05
M3_MACRO_F1_TOL = 0.02
M3_THRESHOLD_POLICY = "low_fpr_enforced"  # {"low_fpr_guarded", "low_fpr_enforced"}

m1 = XGBClassifier(
    objective="binary:logistic",
    n_estimators=400,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=2,
    reg_lambda=2.0,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

m1.fit(X_lex_train, y_train, eval_set=[(X_lex_val, y_val)], verbose=False)

m1_val_proba = m1.predict_proba(X_lex_val)[:, 1]
M1_T_STAR, M1_T_META = best_threshold_low_fpr_with_macro_guard(
    y_val,
    m1_val_proba,
    target_fpr=M3_TARGET_FPR,
    macro_f1_tolerance=M3_MACRO_F1_TOL,
    enforce_target_fpr=(M3_THRESHOLD_POLICY == "low_fpr_enforced"),
    fallback_to_macro_f1=True,
    n_grid=1001,
)

print(f"M1 threshold policy: {M3_THRESHOLD_POLICY}")
print(f"M1 VAL threshold t* = {M1_T_STAR:.3f}")
print("M1 threshold diagnostics:", M1_T_META)

m3 = XGBClassifier(
    objective="binary:logistic",
    n_estimators=400,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=2,
    reg_lambda=2.0,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

m3.fit(X_m3_train, y_train, eval_set=[(X_m3_val, y_val)], verbose=False)

m3_val_proba = m3.predict_proba(X_m3_val)[:, 1]
M3_T_STAR, M3_T_META = best_threshold_low_fpr_with_macro_guard(
    y_val,
    m3_val_proba,
    target_fpr=M3_TARGET_FPR,
    macro_f1_tolerance=M3_MACRO_F1_TOL,
    enforce_target_fpr=(M3_THRESHOLD_POLICY == "low_fpr_enforced"),
    fallback_to_macro_f1=True,
    n_grid=1001,
)

print(f"M3 threshold policy: {M3_THRESHOLD_POLICY}")
print(f"M3 VAL threshold t* = {M3_T_STAR:.3f}")
print("M3 threshold diagnostics:", M3_T_META)



M1 threshold policy: low_fpr_enforced
M1 VAL threshold t* = 0.983
M1 threshold diagnostics: {'selection_mode': 'low_fpr_enforced_macro_best', 'target_fpr': 0.05, 'macro_f1_tolerance': 0.02, 'enforce_target_fpr': True, 'fallback_to_macro_f1': True, 'val_macro_f1_best': 0.8508010680907877, 'val_macro_f1_at_t_star': 0.5300108147080029, 'val_fpr_at_t_star': 0.05, 'val_tpr_at_t_star': 0.3761467889908257, 'val_fpr_constraint_satisfied': True, 'val_fpr_gap_to_target': 0.0, 'val_fpr_min_possible': 0.0, 'val_num_feasible_thresholds': 18}


M3 threshold policy: low_fpr_enforced
M3 VAL threshold t* = 0.959
M3 threshold diagnostics: {'selection_mode': 'low_fpr_enforced_macro_best', 'target_fpr': 0.05, 'macro_f1_tolerance': 0.02, 'enforce_target_fpr': True, 'fallback_to_macro_f1': True, 'val_macro_f1_best': 0.8613265496060061, 'val_macro_f1_at_t_star': 0.7176960970064419, 'val_fpr_at_t_star': 0.05, 'val_tpr_at_t_star': 0.6513761467889908, 'val_fpr_constraint_satisfied': True, 'val_fpr_gap_to_target': 0.0, 'val_fpr_min_possible': 0.0, 'val_num_feasible_thresholds': 42}


In [8]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 5) Train Semantic Model (BGE + Logistic Regression)

sem_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

X_sem_train = encode_texts(sem_model, df_train["prompt_text"].astype(str).tolist())
X_sem_val   = encode_texts(sem_model, df_val["prompt_text"].astype(str).tolist())
X_sem_test  = encode_texts(sem_model, df_test["prompt_text"].astype(str).tolist())

X_sem_ood_map = {
    split_name: encode_texts(sem_model, df_ood_map[split_name]["prompt_text"].astype(str).tolist())
    for split_name in df_ood_map
}
X_sem_ood = X_sem_ood_map["ood_test"]

sem_clf = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
sem_clf.fit(X_sem_train, y_train)

print("Semantic embedding shapes:")
print(X_sem_train.shape, X_sem_val.shape, X_sem_test.shape, X_sem_ood.shape)
for split_name, X in X_sem_ood_map.items():
    print(f"{split_name:18s}", X.shape)


Semantic embedding shapes:
(694, 384) (149, 384) (149, 384) (768, 384)
ood_test           (768, 384)
ood_test_injection (678, 384)
ood_test_injection_standard (3986, 384)


In [9]:
# Cell Purpose: Build or load semantic feature representations for baseline/hybrid models.
# 6) Hybrid Routing + score fusion utilities

LEGACY_IBVS_GATE_DEFINITION = (
    "tripwire_alert>0 OR interaction_system_hierarchy_spoof_chain>0 "
    "OR (hierarchy_override>0 AND system_spoof>0)"
)

HIGH_SPEC_IBVS_GATE_DEFINITION = (
    "high_specific_risk_anchor>0 OR interaction_harm_evasion>0 OR (harm_domain>0 AND evasion>0) "
    "AND benign/meta suppressors are zero"
)



def ibvs_high_precision_mask(ibvs_df: pd.DataFrame) -> np.ndarray:
    if ibvs_df is None or len(ibvs_df) == 0:
        return np.zeros(0, dtype=bool)

    tripwire = ibvs_df.get("tripwire_alert", pd.Series(0.0, index=ibvs_df.index)).astype(float).to_numpy() > 0.0
    spoof_chain = ibvs_df.get(
        "interaction_system_hierarchy_spoof_chain",
        pd.Series(0.0, index=ibvs_df.index),
    ).astype(float).to_numpy() > 0.0
    hierarchy = ibvs_df.get("hierarchy_override", pd.Series(0.0, index=ibvs_df.index)).astype(float).to_numpy() > 0.0
    system_spoof = ibvs_df.get("system_spoof", pd.Series(0.0, index=ibvs_df.index)).astype(float).to_numpy() > 0.0

    return tripwire | spoof_chain | (hierarchy & system_spoof)



def ibvs_high_specific_mask(ibvs_df: pd.DataFrame) -> np.ndarray:
    if ibvs_df is None or len(ibvs_df) == 0:
        return np.zeros(0, dtype=bool)

    anchor = ibvs_df.get("high_specific_risk_anchor", pd.Series(0.0, index=ibvs_df.index)).astype(float).to_numpy() > 0.0
    harm_evasion = ibvs_df.get("interaction_harm_evasion", pd.Series(0.0, index=ibvs_df.index)).astype(float).to_numpy() > 0.0
    harm = ibvs_df.get("harm_domain", pd.Series(0.0, index=ibvs_df.index)).astype(float).to_numpy() > 0.0
    evasion = ibvs_df.get("evasion", pd.Series(0.0, index=ibvs_df.index)).astype(float).to_numpy() > 0.0
    benign_sup = ibvs_df.get("benign_context_suppression", pd.Series(0.0, index=ibvs_df.index)).astype(float).to_numpy() > 0.0
    meta_sup = ibvs_df.get("meta_system_discussion_suppression", pd.Series(0.0, index=ibvs_df.index)).astype(float).to_numpy() > 0.0

    return (anchor | harm_evasion | (harm & evasion)) & ~(benign_sup | meta_sup)



def _fit_calibrators(y_true: np.ndarray, sem_scores: np.ndarray, m3_scores: np.ndarray, method: str):
    if method == "none":
        return (lambda x: np.asarray(x, dtype=float), lambda x: np.asarray(x, dtype=float))
    if method == "isotonic_val":
        sem_iso = IsotonicRegression(out_of_bounds="clip")
        m3_iso = IsotonicRegression(out_of_bounds="clip")
        sem_iso.fit(sem_scores, y_true)
        m3_iso.fit(m3_scores, y_true)
        return (
            lambda x: np.asarray(sem_iso.predict(np.asarray(x, dtype=float)), dtype=float),
            lambda x: np.asarray(m3_iso.predict(np.asarray(x, dtype=float)), dtype=float),
        )
    raise ValueError(f"Unknown calibration method: {method}")



def hybrid_predict(
    X_sem,
    X_m3,
    *,
    tau_low: float,
    tau_high: float,
    m3_threshold: float,
    ibvs_df: pd.DataFrame,
    score_variant: str = "baseline",
    ibvs_boost_alpha: float = 1.0,
    fusion_margin: float = 0.0,
    gate_mode: str = "legacy",
    sem_calibrator=None,
    m3_calibrator=None,
    semantic_threshold: float | None = None,
):
    if not (0.0 <= tau_low < tau_high <= 1.0):
        raise ValueError(f"Require 0 <= tau_low < tau_high <= 1. Got ({tau_low}, {tau_high}).")
    if score_variant not in {"baseline", "anchored_ibvs_boost", "anchored_ibvs_boost_v2", "anchored_ibvs_veto_v1"}:
        raise ValueError(f"Unknown score_variant: {score_variant}")
    if gate_mode not in {"legacy", "high_specific"}:
        raise ValueError(f"Unknown gate_mode: {gate_mode}")
    if ibvs_boost_alpha < 0.0:
        raise ValueError("ibvs_boost_alpha must be >= 0.")
    if fusion_margin < 0.0:
        raise ValueError("fusion_margin must be >= 0.")

    p_sem_raw = sem_clf.predict_proba(X_sem)[:, 1]
    p_m3_raw = m3.predict_proba(X_m3)[:, 1]

    sem_cal = sem_calibrator if sem_calibrator is not None else (lambda x: np.asarray(x, dtype=float))
    m3_cal = m3_calibrator if m3_calibrator is not None else (lambda x: np.asarray(x, dtype=float))

    p_sem_score = np.clip(sem_cal(p_sem_raw), 0.0, 1.0)
    p_m3_score = np.clip(m3_cal(p_m3_raw), 0.0, 1.0)

    y_m3 = (p_m3_raw >= m3_threshold).astype(int)

    y_hat = np.empty_like(y_m3)
    route = np.empty_like(y_m3, dtype=object)

    mask_benign = p_sem_raw <= tau_low
    mask_harm = p_sem_raw >= tau_high
    mask_uncertain = ~(mask_benign | mask_harm)

    y_hat[mask_benign] = 0
    route[mask_benign] = "semantic_low"

    y_hat[mask_harm] = 1
    route[mask_harm] = "semantic_high"

    if score_variant == "anchored_ibvs_veto_v1":
        if semantic_threshold is None:
            raise ValueError("semantic_threshold must be provided for anchored_ibvs_veto_v1")
        y_sem = (p_sem_raw >= semantic_threshold).astype(int)
        y_hat[mask_uncertain] = y_sem[mask_uncertain]
        route[mask_uncertain] = "semantic_uncertain"
    else:
        y_hat[mask_uncertain] = y_m3[mask_uncertain]
        route[mask_uncertain] = "fallback_m3"

    if score_variant == "baseline":
        hybrid_score = p_sem_score.copy()
        hybrid_score[mask_uncertain] = p_m3_score[mask_uncertain]
    else:
        if gate_mode == "legacy":
            gate_mask = ibvs_high_precision_mask(ibvs_df)
        else:
            gate_mask = ibvs_high_specific_mask(ibvs_df)

        if gate_mask.shape[0] != p_sem_score.shape[0]:
            raise ValueError("ibvs_df row count must match X_sem/X_m3 row count.")

        hybrid_score = p_sem_score.copy()
        delta = p_m3_score - p_sem_score

        if score_variant == "anchored_ibvs_boost":
            boost_mask = mask_uncertain & gate_mask & (delta > 0.0)
            hybrid_score[boost_mask] = p_sem_score[boost_mask] + ibvs_boost_alpha * delta[boost_mask]
        elif score_variant == "anchored_ibvs_boost_v2":
            boost_mask = mask_uncertain & gate_mask & (delta > fusion_margin)
            hybrid_score[boost_mask] = (
                p_sem_score[boost_mask] + ibvs_boost_alpha * np.maximum(delta[boost_mask] - fusion_margin, 0.0)
            )
        else:
            # Semantic-anchored veto architecture:
            # keep semantic uncertain decisions by default, allow M3 to veto only when
            # high-specific IBVS cues and strong M3 confidence are present.
            veto_block_mask = mask_uncertain & gate_mask & (p_m3_raw >= (m3_threshold + fusion_margin))
            y_hat[veto_block_mask] = 1
            route[veto_block_mask] = "fallback_veto_block"

            boost_mask = veto_block_mask & (delta > 0.0)
            hybrid_score[boost_mask] = p_sem_score[boost_mask] + ibvs_boost_alpha * delta[boost_mask]

    hybrid_score = np.clip(hybrid_score, 0.0, 1.0)
    if not np.isfinite(hybrid_score).all():
        raise ValueError("hybrid_score contains NaN/inf.")

    return y_hat, hybrid_score, route, p_sem_raw, p_m3_raw


LEARNED_META_IBVS_COMPONENTS = [
    "ibvs_v2_total",
    "tripwire_alert",
    "high_specific_risk_anchor",
    "interaction_harm_evasion",
    "interaction_system_hierarchy_spoof_chain",
    "harm_domain",
    "evasion",
    "hierarchy_override",
    "system_spoof",
    "procedural",
    "tool_directive",
    "benign_context_suppression",
    "meta_system_discussion_suppression",
]
LEARNED_META_FEATURE_DEFINITION = (
    "features=[p_sem_cal,p_m3_cal,delta,abs_delta," + ",".join(LEARNED_META_IBVS_COMPONENTS) + "]"
)


def _ibvs_component_array(ibvs_df: pd.DataFrame, col: str) -> np.ndarray:
    if ibvs_df is None or len(ibvs_df) == 0:
        return np.zeros(0, dtype=float)
    return ibvs_df.get(col, pd.Series(0.0, index=ibvs_df.index)).astype(float).to_numpy()


def build_learned_meta_features(
    p_sem_cal: np.ndarray,
    p_m3_cal: np.ndarray,
    ibvs_df: pd.DataFrame,
):
    p_sem_cal = np.asarray(p_sem_cal, dtype=float)
    p_m3_cal = np.asarray(p_m3_cal, dtype=float)
    if p_sem_cal.shape[0] != p_m3_cal.shape[0]:
        raise ValueError("p_sem_cal and p_m3_cal must have same length.")
    n = p_sem_cal.shape[0]
    if ibvs_df is None or len(ibvs_df) != n:
        raise ValueError("ibvs_df must be provided with matching row count.")

    delta = p_m3_cal - p_sem_cal
    feats = [
        p_sem_cal,
        p_m3_cal,
        delta,
        np.abs(delta),
    ]
    names = ["p_sem_cal", "p_m3_cal", "delta_m3_minus_sem", "abs_delta_m3_sem"]

    for col in LEARNED_META_IBVS_COMPONENTS:
        feats.append(_ibvs_component_array(ibvs_df, col))
        names.append(col)

    X_meta = np.column_stack(feats).astype(float)
    if not np.isfinite(X_meta).all():
        raise ValueError("Learned-meta features contain NaN/inf.")
    return X_meta, names


def learned_meta_predict(
    X_sem,
    X_m3,
    *,
    tau_low: float,
    tau_high: float,
    meta_model,
    meta_threshold: float,
    ibvs_df: pd.DataFrame,
    sem_calibrator=None,
    m3_calibrator=None,
):
    if not (0.0 <= tau_low < tau_high <= 1.0):
        raise ValueError(f"Require 0 <= tau_low < tau_high <= 1. Got ({tau_low}, {tau_high}).")
    if not (0.0 <= meta_threshold <= 1.0):
        raise ValueError(f"meta_threshold must be in [0,1], got {meta_threshold}")

    p_sem_raw = sem_clf.predict_proba(X_sem)[:, 1]
    p_m3_raw = m3.predict_proba(X_m3)[:, 1]

    sem_cal = sem_calibrator if sem_calibrator is not None else (lambda x: np.asarray(x, dtype=float))
    m3_cal = m3_calibrator if m3_calibrator is not None else (lambda x: np.asarray(x, dtype=float))

    p_sem_score = np.clip(sem_cal(p_sem_raw), 0.0, 1.0)
    p_m3_score = np.clip(m3_cal(p_m3_raw), 0.0, 1.0)

    y_hat = np.empty_like(p_sem_raw, dtype=int)
    route = np.empty_like(p_sem_raw, dtype=object)

    mask_benign = p_sem_raw <= tau_low
    mask_harm = p_sem_raw >= tau_high
    mask_uncertain = ~(mask_benign | mask_harm)

    y_hat[mask_benign] = 0
    route[mask_benign] = "semantic_low"

    y_hat[mask_harm] = 1
    route[mask_harm] = "semantic_high"

    hybrid_score = p_sem_score.copy()
    meta_score = np.full_like(p_sem_score, np.nan, dtype=float)

    idx_uncertain = np.where(mask_uncertain)[0]
    if idx_uncertain.size > 0:
        ibvs_u = ibvs_df.iloc[idx_uncertain].reset_index(drop=True)
        X_meta_u, _ = build_learned_meta_features(
            p_sem_score[idx_uncertain],
            p_m3_score[idx_uncertain],
            ibvs_u,
        )
        p_meta_u = np.clip(meta_model.predict_proba(X_meta_u)[:, 1], 0.0, 1.0)

        meta_score[idx_uncertain] = p_meta_u
        hybrid_score[idx_uncertain] = p_meta_u
        y_hat[idx_uncertain] = (p_meta_u >= meta_threshold).astype(int)
        route[idx_uncertain] = "fallback_meta"

    hybrid_score = np.clip(hybrid_score, 0.0, 1.0)
    if not np.isfinite(hybrid_score).all():
        raise ValueError("hybrid_score contains NaN/inf.")

    return y_hat, hybrid_score, route, p_sem_raw, p_m3_raw, meta_score


EXPERT_GATE_IBVS_COMPONENTS = LEARNED_META_IBVS_COMPONENTS
EXPERT_GATE_FEATURE_DEFINITION = (
    "features=[p_sem_cal,p_m1_cal,p_m3_cal,delta_m1_sem,delta_m3_sem,delta_m3_m1,abs_delta_m1_sem,abs_delta_m3_sem,abs_delta_m3_m1,"
    + ",".join(EXPERT_GATE_IBVS_COMPONENTS)
    + "]"
)


def _fit_m1_calibrator(y_true: np.ndarray, m1_scores: np.ndarray, method: str):
    if method == "none":
        return lambda x: np.asarray(x, dtype=float)
    if method == "isotonic_val":
        m1_iso = IsotonicRegression(out_of_bounds="clip")
        m1_iso.fit(m1_scores, y_true)
        return lambda x: np.asarray(m1_iso.predict(np.asarray(x, dtype=float)), dtype=float)
    raise ValueError(f"Unknown calibration method: {method}")


def build_expert_gate_features(
    p_sem_cal: np.ndarray,
    p_m1_cal: np.ndarray,
    p_m3_cal: np.ndarray,
    ibvs_df: pd.DataFrame,
):
    p_sem_cal = np.asarray(p_sem_cal, dtype=float)
    p_m1_cal = np.asarray(p_m1_cal, dtype=float)
    p_m3_cal = np.asarray(p_m3_cal, dtype=float)

    if not (p_sem_cal.shape[0] == p_m1_cal.shape[0] == p_m3_cal.shape[0]):
        raise ValueError("p_sem_cal, p_m1_cal, p_m3_cal must have same length.")

    n = p_sem_cal.shape[0]
    if ibvs_df is None or len(ibvs_df) != n:
        raise ValueError("ibvs_df must be provided with matching row count.")

    d_m1_sem = p_m1_cal - p_sem_cal
    d_m3_sem = p_m3_cal - p_sem_cal
    d_m3_m1 = p_m3_cal - p_m1_cal

    feats = [
        p_sem_cal,
        p_m1_cal,
        p_m3_cal,
        d_m1_sem,
        d_m3_sem,
        d_m3_m1,
        np.abs(d_m1_sem),
        np.abs(d_m3_sem),
        np.abs(d_m3_m1),
    ]
    names = [
        "p_sem_cal",
        "p_m1_cal",
        "p_m3_cal",
        "delta_m1_minus_sem",
        "delta_m3_minus_sem",
        "delta_m3_minus_m1",
        "abs_delta_m1_sem",
        "abs_delta_m3_sem",
        "abs_delta_m3_m1",
    ]

    for col in EXPERT_GATE_IBVS_COMPONENTS:
        feats.append(_ibvs_component_array(ibvs_df, col))
        names.append(col)

    X_gate = np.column_stack(feats).astype(float)
    if not np.isfinite(X_gate).all():
        raise ValueError("Expert-gate features contain NaN/inf.")
    return X_gate, names


def build_expert_gate_targets(
    y_true: np.ndarray,
    sem_pred: np.ndarray,
    m1_pred: np.ndarray,
    m3_pred: np.ndarray,
    p_sem_cal: np.ndarray,
    p_m1_cal: np.ndarray,
    p_m3_cal: np.ndarray,
) -> np.ndarray:
    y_true = np.asarray(y_true, dtype=int)
    sem_pred = np.asarray(sem_pred, dtype=int)
    m1_pred = np.asarray(m1_pred, dtype=int)
    m3_pred = np.asarray(m3_pred, dtype=int)

    n = y_true.shape[0]
    targets = np.zeros(n, dtype=int)

    conf = np.column_stack([
        np.abs(np.asarray(p_sem_cal, dtype=float) - 0.5),
        np.abs(np.asarray(p_m1_cal, dtype=float) - 0.5),
        np.abs(np.asarray(p_m3_cal, dtype=float) - 0.5),
    ])
    prob_err = np.column_stack([
        np.abs(np.asarray(p_sem_cal, dtype=float) - y_true),
        np.abs(np.asarray(p_m1_cal, dtype=float) - y_true),
        np.abs(np.asarray(p_m3_cal, dtype=float) - y_true),
    ])

    preds = np.column_stack([sem_pred, m1_pred, m3_pred])
    correct = preds == y_true[:, None]

    for i in range(n):
        corr_idx = np.where(correct[i])[0]
        if corr_idx.size > 0:
            best_local = corr_idx[np.argmax(conf[i, corr_idx])]
            targets[i] = int(best_local)
        else:
            targets[i] = int(np.argmin(prob_err[i]))

    return targets


def expert_gate_predict(
    X_sem,
    X_lex,
    X_m3,
    *,
    tau_low: float,
    tau_high: float,
    gate_model,
    gate_confidence: float,
    ibvs_df: pd.DataFrame,
    sem_threshold: float,
    m1_threshold: float,
    m3_threshold: float,
    sem_calibrator=None,
    m1_calibrator=None,
    m3_calibrator=None,
):
    if not (0.0 <= tau_low < tau_high <= 1.0):
        raise ValueError(f"Require 0 <= tau_low < tau_high <= 1. Got ({tau_low}, {tau_high}).")
    if not (0.0 <= gate_confidence <= 1.0):
        raise ValueError(f"gate_confidence must be in [0,1], got {gate_confidence}")

    p_sem_raw = sem_clf.predict_proba(X_sem)[:, 1]
    p_m1_raw = m1.predict_proba(X_lex)[:, 1]
    p_m3_raw = m3.predict_proba(X_m3)[:, 1]

    sem_cal = sem_calibrator if sem_calibrator is not None else (lambda x: np.asarray(x, dtype=float))
    m1_cal = m1_calibrator if m1_calibrator is not None else (lambda x: np.asarray(x, dtype=float))
    m3_cal = m3_calibrator if m3_calibrator is not None else (lambda x: np.asarray(x, dtype=float))

    p_sem_score = np.clip(sem_cal(p_sem_raw), 0.0, 1.0)
    p_m1_score = np.clip(m1_cal(p_m1_raw), 0.0, 1.0)
    p_m3_score = np.clip(m3_cal(p_m3_raw), 0.0, 1.0)

    sem_pred = (p_sem_raw >= sem_threshold).astype(int)
    m1_pred = (p_m1_raw >= m1_threshold).astype(int)
    m3_pred = (p_m3_raw >= m3_threshold).astype(int)

    y_hat = np.empty_like(sem_pred)
    route = np.empty_like(sem_pred, dtype=object)

    mask_benign = p_sem_raw <= tau_low
    mask_harm = p_sem_raw >= tau_high
    mask_uncertain = ~(mask_benign | mask_harm)

    y_hat[mask_benign] = 0
    route[mask_benign] = "semantic_low"

    y_hat[mask_harm] = 1
    route[mask_harm] = "semantic_high"

    hybrid_score = p_sem_score.copy()
    gate_max_prob = np.full_like(p_sem_score, np.nan, dtype=float)

    idx_uncertain = np.where(mask_uncertain)[0]
    if idx_uncertain.size > 0:
        ibvs_u = ibvs_df.iloc[idx_uncertain].reset_index(drop=True)
        X_gate_u, _ = build_expert_gate_features(
            p_sem_score[idx_uncertain],
            p_m1_score[idx_uncertain],
            p_m3_score[idx_uncertain],
            ibvs_u,
        )

        gate_proba_u_raw = np.asarray(gate_model.predict_proba(X_gate_u), dtype=float)
        classes = np.asarray(gate_model.classes_, dtype=int)
        gate_proba_u = np.zeros((gate_proba_u_raw.shape[0], 3), dtype=float)
        gate_proba_u[:, classes] = gate_proba_u_raw

        gate_choice_u = np.argmax(gate_proba_u, axis=1)
        gate_max_u = np.max(gate_proba_u, axis=1)

        sem_fallback_u = gate_max_u < gate_confidence
        gate_choice_u = np.where(sem_fallback_u, 0, gate_choice_u)

        pred_matrix_u = np.column_stack([
            sem_pred[idx_uncertain],
            m1_pred[idx_uncertain],
            m3_pred[idx_uncertain],
        ])
        score_matrix_u = np.column_stack([
            p_sem_score[idx_uncertain],
            p_m1_score[idx_uncertain],
            p_m3_score[idx_uncertain],
        ])

        row_idx = np.arange(idx_uncertain.size)
        y_hat[idx_uncertain] = pred_matrix_u[row_idx, gate_choice_u]
        hybrid_score[idx_uncertain] = score_matrix_u[row_idx, gate_choice_u]
        gate_max_prob[idx_uncertain] = gate_max_u

        route_labels = np.asarray(["fallback_gate_sem", "fallback_gate_m1", "fallback_gate_m3"], dtype=object)
        route[idx_uncertain] = route_labels[gate_choice_u]

    hybrid_score = np.clip(hybrid_score, 0.0, 1.0)
    if not np.isfinite(hybrid_score).all():
        raise ValueError("hybrid_score contains NaN/inf.")

    return y_hat, hybrid_score, route, p_sem_raw, p_m1_raw, p_m3_raw, gate_max_prob



In [10]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 7) Evaluate baseline + ID-VAL-only tuned boosts (no OOD tuning)

print(
    "Protocol note: no OOD calibration/tuning is used; all parameter search in this notebook uses ID VAL only."
)

# Semantic VAL reference for constraints.
sem_val_raw = sem_clf.predict_proba(X_sem_val)[:, 1]
SEM_T_STAR, SEM_VAL_MACRO_F1 = best_threshold_by_macro_f1(y_val, sem_val_raw, n_grid=1001)

# M3 VAL scores (used for optional calibration fitting on VAL only).
m3_val_raw = m3.predict_proba(X_m3_val)[:, 1]

# M1 VAL scores (used for expert-gate variant).
m1_val_raw = m1.predict_proba(X_lex_val)[:, 1]

print(f"Semantic VAL macro-F1 reference: {SEM_VAL_MACRO_F1:.4f} at t*={SEM_T_STAR:.3f}")

# ID-only tuning objective: prioritize low-FPR tail while preserving
# macro quality and avoiding overly aggressive fallback routing.
MIN_SEM_COVERAGE = 0.55
MAX_SEM_COVERAGE = 0.95
OBJ_W_TPR1 = 1.0
OBJ_W_TPR5 = 0.5
OBJ_W_MACRO = 0.5
OBJ_W_COVERAGE = 0.25


# -------------------------
# Baseline hybrid (fixed tau)
# -------------------------
BASE_TAU_LOW = 0.30
BASE_TAU_HIGH = 0.70

baseline_note = (
    f"variant=baseline; tau=({BASE_TAU_LOW:.2f},{BASE_TAU_HIGH:.2f}); "
    f"fallback=M3+IBVSv2; m3_t*={M3_T_STAR:.3f}; "
    f"m3_mode={M3_T_META.get('selection_mode')}; m3_val_fpr={M3_T_META.get('val_fpr_at_t_star', np.nan):.3f}; "
    "score=hybrid_score(p_sem_outside,p_m3_inside_uncertain); tuning=id_val_only"
)

val_pred_b, val_score_b, val_route_b, _, _ = hybrid_predict(
    X_sem_val,
    X_m3_val,
    tau_low=BASE_TAU_LOW,
    tau_high=BASE_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_val_df,
    score_variant="baseline",
    gate_mode="legacy",
)
test_pred_b, test_score_b, test_route_b, _, _ = hybrid_predict(
    X_sem_test,
    X_m3_test,
    tau_low=BASE_TAU_LOW,
    tau_high=BASE_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_test_df,
    score_variant="baseline",
    gate_mode="legacy",
)
ood_pred_b, ood_score_b, ood_route_b, _, _ = hybrid_predict(
    X_sem_ood,
    X_m3_ood,
    tau_low=BASE_TAU_LOW,
    tau_high=BASE_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_ood_df,
    score_variant="baseline",
    gate_mode="legacy",
)

val_res_b = evaluate_predictions("VAL (HYBRID)", y_val, val_pred_b, val_score_b, route=val_route_b, threshold_note=baseline_note)
test_res_b = evaluate_predictions("TEST (HYBRID)", y_test, test_pred_b, test_score_b, route=test_route_b, threshold_note=baseline_note)
ood_res_b = evaluate_predictions("OOD (HYBRID)", y_ood, ood_pred_b, ood_score_b, route=ood_route_b, threshold_note=baseline_note)

hybrid_baseline_df = results_to_dataframe("HYBRID_SEM_GATE_ELSE_M3_IBVS_V2", [val_res_b, test_res_b, ood_res_b])
hybrid_baseline_df["eval_track"] = "deployment_threshold"
hybrid_baseline_df["hybrid_variant"] = "baseline"
hybrid_baseline_df["tau_low"] = float(BASE_TAU_LOW)
hybrid_baseline_df["tau_high"] = float(BASE_TAU_HIGH)
hybrid_baseline_df["score_definition"] = "p_sem outside uncertain; p_m3 inside uncertain"
hybrid_baseline_df["ibvs_boost_alpha"] = 0.0
hybrid_baseline_df["ibvs_high_precision_rule"] = LEGACY_IBVS_GATE_DEFINITION
hybrid_baseline_df["tuning_source"] = "id_val_only"
hybrid_baseline_df["fusion_margin"] = 0.0
hybrid_baseline_df["fusion_alpha"] = 0.0
hybrid_baseline_df["ibvs_gate_definition"] = LEGACY_IBVS_GATE_DEFINITION
hybrid_baseline_df["calibration_method"] = "none"


# -----------------------------------------
# Variant 1: legacy anchored boost (VAL-only)
# -----------------------------------------
GRID_TAU_LOW = [0.25, 0.30, 0.35, 0.40]
GRID_TAU_HIGH = [0.60, 0.65, 0.70, 0.75, 0.80]
GRID_ALPHA = [0.25, 0.50, 0.75, 1.00]

rows_v1 = []
for tl in GRID_TAU_LOW:
    for th in GRID_TAU_HIGH:
        if tl >= th:
            continue
        for alpha in GRID_ALPHA:
            val_pred_g, val_score_g, val_route_g, _, _ = hybrid_predict(
                X_sem_val,
                X_m3_val,
                tau_low=tl,
                tau_high=th,
                m3_threshold=M3_T_STAR,
                ibvs_df=ibvs_val_df,
                score_variant="anchored_ibvs_boost",
                ibvs_boost_alpha=alpha,
                fusion_margin=0.0,
                gate_mode="legacy",
            )

            val_res_g = evaluate_predictions(
                f"VAL (HYBRID BOOST V1 tl={tl:.2f},th={th:.2f},a={alpha:.2f})",
                y_val,
                val_pred_g,
                val_score_g,
                print_report=False,
                route=val_route_g,
                threshold_note="variant=anchored_ibvs_boost; tuning=id_val_only",
            )

            j_score = (
                OBJ_W_TPR1 * val_res_g.tpr_at_1pct_fpr
                + OBJ_W_TPR5 * val_res_g.tpr_at_5pct_fpr
                + OBJ_W_MACRO * val_res_g.macro_f1
            )
            coverage = float(val_res_g.semantic_coverage) if val_res_g.semantic_coverage is not None else np.nan
            cov_term = coverage if np.isfinite(coverage) else 0.0
            j_score = j_score + OBJ_W_COVERAGE * cov_term
            pass_macro = bool(val_res_g.macro_f1 >= (SEM_VAL_MACRO_F1 - 0.02))
            pass_cov = bool(np.isfinite(coverage) and (MIN_SEM_COVERAGE <= coverage <= MAX_SEM_COVERAGE))

            rows_v1.append(
                {
                    "tau_low": float(tl),
                    "tau_high": float(th),
                    "alpha": float(alpha),
                    "val_macro_f1": float(val_res_g.macro_f1),
                    "val_tpr_at_1pct_fpr": float(val_res_g.tpr_at_1pct_fpr),
                    "val_tpr_at_5pct_fpr": float(val_res_g.tpr_at_5pct_fpr),
                    "semantic_coverage": coverage,
                    "objective_j": float(j_score),
                    "is_feasible": bool(pass_macro and pass_cov),
                }
            )

tune_v1 = pd.DataFrame(rows_v1)
feasible_v1 = tune_v1[tune_v1["is_feasible"]].copy()
if feasible_v1.empty:
    best_v1 = tune_v1.sort_values(by=["objective_j", "val_macro_f1", "semantic_coverage"], ascending=False).iloc[0]
else:
    best_v1 = feasible_v1.sort_values(by=["objective_j", "val_macro_f1", "semantic_coverage"], ascending=False).iloc[0]

BOOST1_TAU_LOW = float(best_v1["tau_low"])
BOOST1_TAU_HIGH = float(best_v1["tau_high"])
BOOST1_ALPHA = float(best_v1["alpha"])

boost1_note = (
    f"variant=anchored_ibvs_boost; tau=({BOOST1_TAU_LOW:.2f},{BOOST1_TAU_HIGH:.2f}); alpha={BOOST1_ALPHA:.2f}; "
    f"fallback=M3+IBVSv2; m3_t*={M3_T_STAR:.3f}; m3_val_fpr={M3_T_META.get('val_fpr_at_t_star', np.nan):.3f}; "
    f"gate={LEGACY_IBVS_GATE_DEFINITION}; tuning=id_val_only"
)

val_pred_u, val_score_u, val_route_u, _, _ = hybrid_predict(
    X_sem_val, X_m3_val,
    tau_low=BOOST1_TAU_LOW,
    tau_high=BOOST1_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_val_df,
    score_variant="anchored_ibvs_boost",
    ibvs_boost_alpha=BOOST1_ALPHA,
    fusion_margin=0.0,
    gate_mode="legacy",
)
test_pred_u, test_score_u, test_route_u, _, _ = hybrid_predict(
    X_sem_test, X_m3_test,
    tau_low=BOOST1_TAU_LOW,
    tau_high=BOOST1_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_test_df,
    score_variant="anchored_ibvs_boost",
    ibvs_boost_alpha=BOOST1_ALPHA,
    fusion_margin=0.0,
    gate_mode="legacy",
)
ood_pred_u, ood_score_u, ood_route_u, _, _ = hybrid_predict(
    X_sem_ood, X_m3_ood,
    tau_low=BOOST1_TAU_LOW,
    tau_high=BOOST1_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_ood_df,
    score_variant="anchored_ibvs_boost",
    ibvs_boost_alpha=BOOST1_ALPHA,
    fusion_margin=0.0,
    gate_mode="legacy",
)

val_res_u = evaluate_predictions("VAL (HYBRID)", y_val, val_pred_u, val_score_u, route=val_route_u, threshold_note=boost1_note)
test_res_u = evaluate_predictions("TEST (HYBRID)", y_test, test_pred_u, test_score_u, route=test_route_u, threshold_note=boost1_note)
ood_res_u = evaluate_predictions("OOD (HYBRID)", y_ood, ood_pred_u, ood_score_u, route=ood_route_u, threshold_note=boost1_note)

hybrid_boost_df = results_to_dataframe("HYBRID_SEM_ANCHORED_IBVS_BOOST", [val_res_u, test_res_u, ood_res_u])
hybrid_boost_df["eval_track"] = "deployment_threshold"
hybrid_boost_df["hybrid_variant"] = "anchored_ibvs_boost"
hybrid_boost_df["tau_low"] = float(BOOST1_TAU_LOW)
hybrid_boost_df["tau_high"] = float(BOOST1_TAU_HIGH)
hybrid_boost_df["score_definition"] = "p_sem globally; uncertain boosted toward p_m3 where legacy ibvs gate and delta>0"
hybrid_boost_df["ibvs_boost_alpha"] = float(BOOST1_ALPHA)
hybrid_boost_df["ibvs_high_precision_rule"] = LEGACY_IBVS_GATE_DEFINITION
hybrid_boost_df["tuning_source"] = "id_val_only"
hybrid_boost_df["fusion_margin"] = 0.0
hybrid_boost_df["fusion_alpha"] = float(BOOST1_ALPHA)
hybrid_boost_df["ibvs_gate_definition"] = LEGACY_IBVS_GATE_DEFINITION
hybrid_boost_df["calibration_method"] = "none"


# ---------------------------------------------------------
# Variant 2: high-specific gate + optional VAL calibration
# ---------------------------------------------------------
GRID_MARGIN = [0.00, 0.02, 0.05, 0.10]
GRID_CAL_METHOD = ["none", "isotonic_val"]

rows_v2 = []
for tl in GRID_TAU_LOW:
    for th in GRID_TAU_HIGH:
        if tl >= th:
            continue
        for alpha in GRID_ALPHA:
            for margin in GRID_MARGIN:
                for cal_method in GRID_CAL_METHOD:
                    sem_cal, m3_cal = _fit_calibrators(y_val, sem_val_raw, m3_val_raw, method=cal_method)
                    val_pred_v2, val_score_v2, val_route_v2, _, _ = hybrid_predict(
                        X_sem_val,
                        X_m3_val,
                        tau_low=tl,
                        tau_high=th,
                        m3_threshold=M3_T_STAR,
                        ibvs_df=ibvs_val_df,
                        score_variant="anchored_ibvs_boost_v2",
                        ibvs_boost_alpha=alpha,
                        fusion_margin=margin,
                        gate_mode="high_specific",
                        sem_calibrator=sem_cal,
                        m3_calibrator=m3_cal,
                    )

                    val_res_v2 = evaluate_predictions(
                        f"VAL (HYBRID BOOST V2 tl={tl:.2f},th={th:.2f},a={alpha:.2f},m={margin:.2f},c={cal_method})",
                        y_val,
                        val_pred_v2,
                        val_score_v2,
                        print_report=False,
                        route=val_route_v2,
                        threshold_note="variant=anchored_ibvs_boost_v2; tuning=id_val_only",
                    )

                    j_score = (
                        OBJ_W_TPR1 * val_res_v2.tpr_at_1pct_fpr
                        + OBJ_W_TPR5 * val_res_v2.tpr_at_5pct_fpr
                        + OBJ_W_MACRO * val_res_v2.macro_f1
                    )
                    coverage = float(val_res_v2.semantic_coverage) if val_res_v2.semantic_coverage is not None else np.nan
                    cov_term = coverage if np.isfinite(coverage) else 0.0
                    j_score = j_score + OBJ_W_COVERAGE * cov_term
                    pass_macro = bool(val_res_v2.macro_f1 >= (SEM_VAL_MACRO_F1 - 0.02))
                    pass_cov = bool(np.isfinite(coverage) and (MIN_SEM_COVERAGE <= coverage <= MAX_SEM_COVERAGE))

                    rows_v2.append(
                        {
                            "tau_low": float(tl),
                            "tau_high": float(th),
                            "alpha": float(alpha),
                            "margin": float(margin),
                            "calibration_method": cal_method,
                            "val_macro_f1": float(val_res_v2.macro_f1),
                            "val_tpr_at_1pct_fpr": float(val_res_v2.tpr_at_1pct_fpr),
                            "val_tpr_at_5pct_fpr": float(val_res_v2.tpr_at_5pct_fpr),
                            "semantic_coverage": coverage,
                            "objective_j": float(j_score),
                            "is_feasible": bool(pass_macro and pass_cov),
                        }
                    )

tune_v2 = pd.DataFrame(rows_v2)
feasible_v2 = tune_v2[tune_v2["is_feasible"]].copy()
if feasible_v2.empty:
    best_v2 = tune_v2.sort_values(by=["objective_j", "val_macro_f1", "semantic_coverage"], ascending=False).iloc[0]
else:
    best_v2 = feasible_v2.sort_values(by=["objective_j", "val_macro_f1", "semantic_coverage"], ascending=False).iloc[0]

BOOST2_TAU_LOW = float(best_v2["tau_low"])
BOOST2_TAU_HIGH = float(best_v2["tau_high"])
BOOST2_ALPHA = float(best_v2["alpha"])
BOOST2_MARGIN = float(best_v2["margin"])
BOOST2_CAL_METHOD = str(best_v2["calibration_method"])

sem_cal_v2, m3_cal_v2 = _fit_calibrators(y_val, sem_val_raw, m3_val_raw, method=BOOST2_CAL_METHOD)

boost2_note = (
    f"variant=anchored_ibvs_boost_v2; tau=({BOOST2_TAU_LOW:.2f},{BOOST2_TAU_HIGH:.2f}); "
    f"alpha={BOOST2_ALPHA:.2f}; margin={BOOST2_MARGIN:.2f}; calibration={BOOST2_CAL_METHOD}; "
    f"fallback=M3+IBVSv2; m3_t*={M3_T_STAR:.3f}; m3_val_fpr={M3_T_META.get('val_fpr_at_t_star', np.nan):.3f}; "
    f"gate={HIGH_SPEC_IBVS_GATE_DEFINITION}; tuning=id_val_only"
)

val_pred_v2, val_score_v2, val_route_v2, _, _ = hybrid_predict(
    X_sem_val, X_m3_val,
    tau_low=BOOST2_TAU_LOW,
    tau_high=BOOST2_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_val_df,
    score_variant="anchored_ibvs_boost_v2",
    ibvs_boost_alpha=BOOST2_ALPHA,
    fusion_margin=BOOST2_MARGIN,
    gate_mode="high_specific",
    sem_calibrator=sem_cal_v2,
    m3_calibrator=m3_cal_v2,
)
test_pred_v2, test_score_v2, test_route_v2, _, _ = hybrid_predict(
    X_sem_test, X_m3_test,
    tau_low=BOOST2_TAU_LOW,
    tau_high=BOOST2_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_test_df,
    score_variant="anchored_ibvs_boost_v2",
    ibvs_boost_alpha=BOOST2_ALPHA,
    fusion_margin=BOOST2_MARGIN,
    gate_mode="high_specific",
    sem_calibrator=sem_cal_v2,
    m3_calibrator=m3_cal_v2,
)
ood_pred_v2, ood_score_v2, ood_route_v2, _, _ = hybrid_predict(
    X_sem_ood, X_m3_ood,
    tau_low=BOOST2_TAU_LOW,
    tau_high=BOOST2_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_ood_df,
    score_variant="anchored_ibvs_boost_v2",
    ibvs_boost_alpha=BOOST2_ALPHA,
    fusion_margin=BOOST2_MARGIN,
    gate_mode="high_specific",
    sem_calibrator=sem_cal_v2,
    m3_calibrator=m3_cal_v2,
)

val_res_v2 = evaluate_predictions("VAL (HYBRID)", y_val, val_pred_v2, val_score_v2, route=val_route_v2, threshold_note=boost2_note)
test_res_v2 = evaluate_predictions("TEST (HYBRID)", y_test, test_pred_v2, test_score_v2, route=test_route_v2, threshold_note=boost2_note)
ood_res_v2 = evaluate_predictions("OOD (HYBRID)", y_ood, ood_pred_v2, ood_score_v2, route=ood_route_v2, threshold_note=boost2_note)

hybrid_boost_v2_df = results_to_dataframe("HYBRID_SEM_ANCHORED_IBVS_BOOST_V2", [val_res_v2, test_res_v2, ood_res_v2])
hybrid_boost_v2_df["eval_track"] = "deployment_threshold"
hybrid_boost_v2_df["hybrid_variant"] = "anchored_ibvs_boost_v2"
hybrid_boost_v2_df["tau_low"] = float(BOOST2_TAU_LOW)
hybrid_boost_v2_df["tau_high"] = float(BOOST2_TAU_HIGH)
hybrid_boost_v2_df["score_definition"] = "p_sem_cal globally; uncertain boosted with relu(delta-margin) on high-specific ibvs gate"
hybrid_boost_v2_df["ibvs_boost_alpha"] = float(BOOST2_ALPHA)
hybrid_boost_v2_df["ibvs_high_precision_rule"] = HIGH_SPEC_IBVS_GATE_DEFINITION
hybrid_boost_v2_df["tuning_source"] = "id_val_only"
hybrid_boost_v2_df["fusion_margin"] = float(BOOST2_MARGIN)
hybrid_boost_v2_df["fusion_alpha"] = float(BOOST2_ALPHA)
hybrid_boost_v2_df["ibvs_gate_definition"] = HIGH_SPEC_IBVS_GATE_DEFINITION
hybrid_boost_v2_df["calibration_method"] = BOOST2_CAL_METHOD



# ---------------------------------------------------------
# Variant 3: semantic-anchored veto (single architectural change)
# ---------------------------------------------------------
GRID_VETO_MARGIN = [0.00, 0.05, 0.10, 0.15]

rows_veto = []
for tl in GRID_TAU_LOW:
    for th in GRID_TAU_HIGH:
        if tl >= th:
            continue
        for alpha in GRID_ALPHA:
            for veto_margin in GRID_VETO_MARGIN:
                for cal_method in GRID_CAL_METHOD:
                    sem_cal, m3_cal = _fit_calibrators(y_val, sem_val_raw, m3_val_raw, method=cal_method)
                    val_pred_veto, val_score_veto, val_route_veto, _, _ = hybrid_predict(
                        X_sem_val,
                        X_m3_val,
                        tau_low=tl,
                        tau_high=th,
                        m3_threshold=M3_T_STAR,
                        ibvs_df=ibvs_val_df,
                        score_variant="anchored_ibvs_veto_v1",
                        ibvs_boost_alpha=alpha,
                        fusion_margin=veto_margin,
                        gate_mode="high_specific",
                        sem_calibrator=sem_cal,
                        m3_calibrator=m3_cal,
                        semantic_threshold=SEM_T_STAR,
                    )

                    val_res_veto = evaluate_predictions(
                        f"VAL (HYBRID VETO tl={tl:.2f},th={th:.2f},a={alpha:.2f},vm={veto_margin:.2f},c={cal_method})",
                        y_val,
                        val_pred_veto,
                        val_score_veto,
                        print_report=False,
                        route=val_route_veto,
                        threshold_note="variant=anchored_ibvs_veto_v1; tuning=id_val_only",
                    )

                    j_score = (
                        OBJ_W_TPR1 * val_res_veto.tpr_at_1pct_fpr
                        + OBJ_W_TPR5 * val_res_veto.tpr_at_5pct_fpr
                        + OBJ_W_MACRO * val_res_veto.macro_f1
                    )
                    coverage = float(val_res_veto.semantic_coverage) if val_res_veto.semantic_coverage is not None else np.nan
                    cov_term = coverage if np.isfinite(coverage) else 0.0
                    j_score = j_score + OBJ_W_COVERAGE * cov_term
                    pass_macro = bool(val_res_veto.macro_f1 >= (SEM_VAL_MACRO_F1 - 0.01))
                    pass_cov = bool(np.isfinite(coverage) and (MIN_SEM_COVERAGE <= coverage <= MAX_SEM_COVERAGE))

                    rows_veto.append(
                        {
                            "tau_low": float(tl),
                            "tau_high": float(th),
                            "alpha": float(alpha),
                            "veto_margin": float(veto_margin),
                            "calibration_method": cal_method,
                            "val_macro_f1": float(val_res_veto.macro_f1),
                            "val_tpr_at_1pct_fpr": float(val_res_veto.tpr_at_1pct_fpr),
                            "val_tpr_at_5pct_fpr": float(val_res_veto.tpr_at_5pct_fpr),
                            "semantic_coverage": coverage,
                            "objective_j": float(j_score),
                            "is_feasible": bool(pass_macro and pass_cov),
                        }
                    )

tune_veto = pd.DataFrame(rows_veto)
feasible_veto = tune_veto[tune_veto["is_feasible"]].copy()
if feasible_veto.empty:
    best_veto = tune_veto.sort_values(by=["objective_j", "val_macro_f1", "semantic_coverage"], ascending=False).iloc[0]
else:
    best_veto = feasible_veto.sort_values(by=["objective_j", "val_macro_f1", "semantic_coverage"], ascending=False).iloc[0]

VETO_TAU_LOW = float(best_veto["tau_low"])
VETO_TAU_HIGH = float(best_veto["tau_high"])
VETO_ALPHA = float(best_veto["alpha"])
VETO_MARGIN = float(best_veto["veto_margin"])
VETO_CAL_METHOD = str(best_veto["calibration_method"])

sem_cal_veto, m3_cal_veto = _fit_calibrators(y_val, sem_val_raw, m3_val_raw, method=VETO_CAL_METHOD)

veto_note = (
    f"variant=anchored_ibvs_veto_v1; tau=({VETO_TAU_LOW:.2f},{VETO_TAU_HIGH:.2f}); "
    f"alpha={VETO_ALPHA:.2f}; veto_margin={VETO_MARGIN:.2f}; calibration={VETO_CAL_METHOD}; "
    f"fallback=M3+IBVSv2; m3_t*={M3_T_STAR:.3f}; sem_t*={SEM_T_STAR:.3f}; "
    f"m3_val_fpr={M3_T_META.get('val_fpr_at_t_star', np.nan):.3f}; gate={HIGH_SPEC_IBVS_GATE_DEFINITION}; tuning=id_val_only"
)

val_pred_veto, val_score_veto, val_route_veto, _, _ = hybrid_predict(
    X_sem_val, X_m3_val,
    tau_low=VETO_TAU_LOW,
    tau_high=VETO_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_val_df,
    score_variant="anchored_ibvs_veto_v1",
    ibvs_boost_alpha=VETO_ALPHA,
    fusion_margin=VETO_MARGIN,
    gate_mode="high_specific",
    sem_calibrator=sem_cal_veto,
    m3_calibrator=m3_cal_veto,
    semantic_threshold=SEM_T_STAR,
)
test_pred_veto, test_score_veto, test_route_veto, _, _ = hybrid_predict(
    X_sem_test, X_m3_test,
    tau_low=VETO_TAU_LOW,
    tau_high=VETO_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_test_df,
    score_variant="anchored_ibvs_veto_v1",
    ibvs_boost_alpha=VETO_ALPHA,
    fusion_margin=VETO_MARGIN,
    gate_mode="high_specific",
    sem_calibrator=sem_cal_veto,
    m3_calibrator=m3_cal_veto,
    semantic_threshold=SEM_T_STAR,
)
ood_pred_veto, ood_score_veto, ood_route_veto, _, _ = hybrid_predict(
    X_sem_ood, X_m3_ood,
    tau_low=VETO_TAU_LOW,
    tau_high=VETO_TAU_HIGH,
    m3_threshold=M3_T_STAR,
    ibvs_df=ibvs_ood_df,
    score_variant="anchored_ibvs_veto_v1",
    ibvs_boost_alpha=VETO_ALPHA,
    fusion_margin=VETO_MARGIN,
    gate_mode="high_specific",
    sem_calibrator=sem_cal_veto,
    m3_calibrator=m3_cal_veto,
    semantic_threshold=SEM_T_STAR,
)

val_res_veto = evaluate_predictions("VAL (HYBRID)", y_val, val_pred_veto, val_score_veto, route=val_route_veto, threshold_note=veto_note)
test_res_veto = evaluate_predictions("TEST (HYBRID)", y_test, test_pred_veto, test_score_veto, route=test_route_veto, threshold_note=veto_note)
ood_res_veto = evaluate_predictions("OOD (HYBRID)", y_ood, ood_pred_veto, ood_score_veto, route=ood_route_veto, threshold_note=veto_note)

hybrid_veto_df = results_to_dataframe("HYBRID_SEM_ANCHORED_IBVS_VETO", [val_res_veto, test_res_veto, ood_res_veto])
hybrid_veto_df["eval_track"] = "deployment_threshold"
hybrid_veto_df["hybrid_variant"] = "anchored_ibvs_veto_v1"
hybrid_veto_df["tau_low"] = float(VETO_TAU_LOW)
hybrid_veto_df["tau_high"] = float(VETO_TAU_HIGH)
hybrid_veto_df["score_definition"] = "semantic-anchored uncertain routing; M3 veto-to-block only on high-specific IBVS gate + confidence margin"
hybrid_veto_df["ibvs_boost_alpha"] = float(VETO_ALPHA)
hybrid_veto_df["ibvs_high_precision_rule"] = HIGH_SPEC_IBVS_GATE_DEFINITION
hybrid_veto_df["tuning_source"] = "id_val_only"
hybrid_veto_df["fusion_margin"] = float(VETO_MARGIN)
hybrid_veto_df["fusion_alpha"] = float(VETO_ALPHA)
hybrid_veto_df["ibvs_gate_definition"] = HIGH_SPEC_IBVS_GATE_DEFINITION
hybrid_veto_df["calibration_method"] = VETO_CAL_METHOD


# ----------------------------------------------------------------
# Variant 4: learned uncertain fusion (ID-only meta-model)
# ----------------------------------------------------------------
GRID_META_C = [0.05, 0.10, 0.50, 1.00, 2.00]
GRID_META_THRESHOLD = [0.40, 0.50, 0.60]

rows_meta = []
for tl in GRID_TAU_LOW:
    for th in GRID_TAU_HIGH:
        if tl >= th:
            continue

        mask_val_uncertain = (sem_val_raw > tl) & (sem_val_raw < th)
        idx_val_uncertain = np.where(mask_val_uncertain)[0]
        if idx_val_uncertain.size < 20:
            continue

        y_val_u = y_val[idx_val_uncertain]
        if np.unique(y_val_u).size < 2:
            continue

        for cal_method in GRID_CAL_METHOD:
            sem_cal, m3_cal = _fit_calibrators(y_val, sem_val_raw, m3_val_raw, method=cal_method)
            p_sem_val_cal = np.clip(sem_cal(sem_val_raw), 0.0, 1.0)
            p_m3_val_cal = np.clip(m3_cal(m3_val_raw), 0.0, 1.0)

            ibvs_val_u = ibvs_val_df.iloc[idx_val_uncertain].reset_index(drop=True)
            X_meta_val_u, _ = build_learned_meta_features(
                p_sem_val_cal[idx_val_uncertain],
                p_m3_val_cal[idx_val_uncertain],
                ibvs_val_u,
            )

            for c_val in GRID_META_C:
                meta_model = LogisticRegression(
                    max_iter=2000,
                    C=float(c_val),
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=RANDOM_SEED,
                )
                meta_model.fit(X_meta_val_u, y_val_u)
                p_meta_val_u = np.clip(meta_model.predict_proba(X_meta_val_u)[:, 1], 0.0, 1.0)

                for meta_t in GRID_META_THRESHOLD:
                    val_pred_meta = np.empty_like(y_val)
                    val_route_meta = np.empty_like(y_val, dtype=object)

                    mask_b = sem_val_raw <= tl
                    mask_h = sem_val_raw >= th
                    mask_u = ~(mask_b | mask_h)

                    val_pred_meta[mask_b] = 0
                    val_route_meta[mask_b] = "semantic_low"

                    val_pred_meta[mask_h] = 1
                    val_route_meta[mask_h] = "semantic_high"

                    val_pred_meta[mask_u] = (p_meta_val_u >= meta_t).astype(int)
                    val_route_meta[mask_u] = "fallback_meta"

                    val_score_meta = p_sem_val_cal.copy()
                    val_score_meta[mask_u] = p_meta_val_u

                    val_res_meta = evaluate_predictions(
                        f"VAL (HYBRID LEARNED tl={tl:.2f},th={th:.2f},C={c_val:.2f},mt={meta_t:.2f},c={cal_method})",
                        y_val,
                        val_pred_meta,
                        val_score_meta,
                        print_report=False,
                        route=val_route_meta,
                        threshold_note="variant=learned_meta_fusion_v1; tuning=id_val_only",
                    )

                    j_score = (
                        OBJ_W_TPR1 * val_res_meta.tpr_at_1pct_fpr
                        + OBJ_W_TPR5 * val_res_meta.tpr_at_5pct_fpr
                        + OBJ_W_MACRO * val_res_meta.macro_f1
                    )
                    coverage = float(val_res_meta.semantic_coverage) if val_res_meta.semantic_coverage is not None else np.nan
                    cov_term = coverage if np.isfinite(coverage) else 0.0
                    j_score = j_score + OBJ_W_COVERAGE * cov_term
                    pass_macro = bool(val_res_meta.macro_f1 >= (SEM_VAL_MACRO_F1 - 0.01))
                    pass_cov = bool(np.isfinite(coverage) and (MIN_SEM_COVERAGE <= coverage <= MAX_SEM_COVERAGE))

                    rows_meta.append(
                        {
                            "tau_low": float(tl),
                            "tau_high": float(th),
                            "meta_c": float(c_val),
                            "meta_threshold": float(meta_t),
                            "calibration_method": cal_method,
                            "val_macro_f1": float(val_res_meta.macro_f1),
                            "val_tpr_at_1pct_fpr": float(val_res_meta.tpr_at_1pct_fpr),
                            "val_tpr_at_5pct_fpr": float(val_res_meta.tpr_at_5pct_fpr),
                            "semantic_coverage": coverage,
                            "objective_j": float(j_score),
                            "n_uncertain": int(idx_val_uncertain.size),
                            "is_feasible": bool(pass_macro and pass_cov),
                        }
                    )

if len(rows_meta) == 0:
    raise ValueError("No feasible candidates for learned meta fusion; uncertain region too small or single-class.")

tune_meta = pd.DataFrame(rows_meta)
feasible_meta = tune_meta[tune_meta["is_feasible"]].copy()
if feasible_meta.empty:
    best_meta = tune_meta.sort_values(by=["objective_j", "val_macro_f1", "semantic_coverage"], ascending=False).iloc[0]
else:
    best_meta = feasible_meta.sort_values(by=["objective_j", "val_macro_f1", "semantic_coverage"], ascending=False).iloc[0]

META_TAU_LOW = float(best_meta["tau_low"])
META_TAU_HIGH = float(best_meta["tau_high"])
META_C = float(best_meta["meta_c"])
META_THRESHOLD = float(best_meta["meta_threshold"])
META_CAL_METHOD = str(best_meta["calibration_method"])

sem_cal_meta, m3_cal_meta = _fit_calibrators(y_val, sem_val_raw, m3_val_raw, method=META_CAL_METHOD)

mask_val_uncertain_best = (sem_val_raw > META_TAU_LOW) & (sem_val_raw < META_TAU_HIGH)
idx_val_uncertain_best = np.where(mask_val_uncertain_best)[0]
if idx_val_uncertain_best.size < 20:
    raise ValueError("Learned meta fusion selected degenerate uncertain region (<20 samples).")

y_val_u_best = y_val[idx_val_uncertain_best]
if np.unique(y_val_u_best).size < 2:
    raise ValueError("Learned meta fusion selected uncertain region with a single class.")

p_sem_val_cal_best = np.clip(sem_cal_meta(sem_val_raw), 0.0, 1.0)
p_m3_val_cal_best = np.clip(m3_cal_meta(m3_val_raw), 0.0, 1.0)

ibvs_val_u_best = ibvs_val_df.iloc[idx_val_uncertain_best].reset_index(drop=True)
X_meta_val_u_best, _ = build_learned_meta_features(
    p_sem_val_cal_best[idx_val_uncertain_best],
    p_m3_val_cal_best[idx_val_uncertain_best],
    ibvs_val_u_best,
)

meta_model_best = LogisticRegression(
    max_iter=2000,
    C=META_C,
    class_weight="balanced",
    solver="liblinear",
    random_state=RANDOM_SEED,
)
meta_model_best.fit(X_meta_val_u_best, y_val_u_best)

meta_note = (
    f"variant=learned_meta_fusion_v1; tau=({META_TAU_LOW:.2f},{META_TAU_HIGH:.2f}); "
    f"meta_C={META_C:.2f}; meta_threshold={META_THRESHOLD:.2f}; calibration={META_CAL_METHOD}; "
    f"m3_t*={M3_T_STAR:.3f}; sem_t*={SEM_T_STAR:.3f}; tuning=id_val_only"
)

val_pred_meta, val_score_meta, val_route_meta, _, _, _ = learned_meta_predict(
    X_sem_val,
    X_m3_val,
    tau_low=META_TAU_LOW,
    tau_high=META_TAU_HIGH,
    meta_model=meta_model_best,
    meta_threshold=META_THRESHOLD,
    ibvs_df=ibvs_val_df,
    sem_calibrator=sem_cal_meta,
    m3_calibrator=m3_cal_meta,
)
test_pred_meta, test_score_meta, test_route_meta, _, _, _ = learned_meta_predict(
    X_sem_test,
    X_m3_test,
    tau_low=META_TAU_LOW,
    tau_high=META_TAU_HIGH,
    meta_model=meta_model_best,
    meta_threshold=META_THRESHOLD,
    ibvs_df=ibvs_test_df,
    sem_calibrator=sem_cal_meta,
    m3_calibrator=m3_cal_meta,
)
ood_pred_meta, ood_score_meta, ood_route_meta, _, _, _ = learned_meta_predict(
    X_sem_ood,
    X_m3_ood,
    tau_low=META_TAU_LOW,
    tau_high=META_TAU_HIGH,
    meta_model=meta_model_best,
    meta_threshold=META_THRESHOLD,
    ibvs_df=ibvs_ood_df,
    sem_calibrator=sem_cal_meta,
    m3_calibrator=m3_cal_meta,
)

val_res_meta = evaluate_predictions("VAL (HYBRID)", y_val, val_pred_meta, val_score_meta, route=val_route_meta, threshold_note=meta_note)
test_res_meta = evaluate_predictions("TEST (HYBRID)", y_test, test_pred_meta, test_score_meta, route=test_route_meta, threshold_note=meta_note)
ood_res_meta = evaluate_predictions("OOD (HYBRID)", y_ood, ood_pred_meta, ood_score_meta, route=ood_route_meta, threshold_note=meta_note)

hybrid_meta_df = results_to_dataframe("HYBRID_SEM_LEARNED_META_FUSION", [val_res_meta, test_res_meta, ood_res_meta])
hybrid_meta_df["eval_track"] = "deployment_threshold"
hybrid_meta_df["hybrid_variant"] = "learned_meta_fusion_v1"
hybrid_meta_df["tau_low"] = float(META_TAU_LOW)
hybrid_meta_df["tau_high"] = float(META_TAU_HIGH)
hybrid_meta_df["score_definition"] = "semantic outside uncertain; uncertain scored by learned logistic fusion of semantic, m3, and ibvs components"
hybrid_meta_df["ibvs_boost_alpha"] = 0.0
hybrid_meta_df["ibvs_high_precision_rule"] = LEARNED_META_FEATURE_DEFINITION
hybrid_meta_df["tuning_source"] = "id_val_only"
hybrid_meta_df["fusion_margin"] = float(META_THRESHOLD)
hybrid_meta_df["fusion_alpha"] = float(META_C)
hybrid_meta_df["ibvs_gate_definition"] = LEARNED_META_FEATURE_DEFINITION
hybrid_meta_df["calibration_method"] = META_CAL_METHOD


# ----------------------------------------------------------------
# Variant 5: learned expert gate on uncertain region (semantic vs M1 vs M3)
# ----------------------------------------------------------------
GRID_GATE_C = [0.10, 0.50, 1.00, 2.00]
GRID_GATE_CONF = [0.34, 0.45, 0.55, 0.65]

rows_gate = []
for tl in GRID_TAU_LOW:
    for th in GRID_TAU_HIGH:
        if tl >= th:
            continue

        mask_val_uncertain = (sem_val_raw > tl) & (sem_val_raw < th)
        idx_val_uncertain = np.where(mask_val_uncertain)[0]
        if idx_val_uncertain.size < 20:
            continue

        y_val_u = y_val[idx_val_uncertain]
        if np.unique(y_val_u).size < 2:
            continue

        for cal_method in GRID_CAL_METHOD:
            sem_cal_gate, m3_cal_gate = _fit_calibrators(y_val, sem_val_raw, m3_val_raw, method=cal_method)
            m1_cal_gate = _fit_m1_calibrator(y_val, m1_val_raw, method=cal_method)

            p_sem_val_cal = np.clip(sem_cal_gate(sem_val_raw), 0.0, 1.0)
            p_m1_val_cal = np.clip(m1_cal_gate(m1_val_raw), 0.0, 1.0)
            p_m3_val_cal = np.clip(m3_cal_gate(m3_val_raw), 0.0, 1.0)

            sem_pred_u = (sem_val_raw[idx_val_uncertain] >= SEM_T_STAR).astype(int)
            m1_pred_u = (m1_val_raw[idx_val_uncertain] >= M1_T_STAR).astype(int)
            m3_pred_u = (m3_val_raw[idx_val_uncertain] >= M3_T_STAR).astype(int)

            y_gate_u = build_expert_gate_targets(
                y_val_u,
                sem_pred_u,
                m1_pred_u,
                m3_pred_u,
                p_sem_val_cal[idx_val_uncertain],
                p_m1_val_cal[idx_val_uncertain],
                p_m3_val_cal[idx_val_uncertain],
            )
            if np.unique(y_gate_u).size < 2:
                continue

            ibvs_val_u = ibvs_val_df.iloc[idx_val_uncertain].reset_index(drop=True)
            X_gate_val_u, _ = build_expert_gate_features(
                p_sem_val_cal[idx_val_uncertain],
                p_m1_val_cal[idx_val_uncertain],
                p_m3_val_cal[idx_val_uncertain],
                ibvs_val_u,
            )

            for c_val in GRID_GATE_C:
                gate_model = LogisticRegression(
                    max_iter=3000,
                    C=float(c_val),
                    class_weight="balanced",
                    solver="lbfgs",
                    multi_class="multinomial",
                    random_state=RANDOM_SEED,
                )
                gate_model.fit(X_gate_val_u, y_gate_u)

                for gate_conf in GRID_GATE_CONF:
                    val_pred_gate, val_score_gate, val_route_gate, _, _, _, _ = expert_gate_predict(
                        X_sem_val,
                        X_lex_val,
                        X_m3_val,
                        tau_low=tl,
                        tau_high=th,
                        gate_model=gate_model,
                        gate_confidence=float(gate_conf),
                        ibvs_df=ibvs_val_df,
                        sem_threshold=SEM_T_STAR,
                        m1_threshold=M1_T_STAR,
                        m3_threshold=M3_T_STAR,
                        sem_calibrator=sem_cal_gate,
                        m1_calibrator=m1_cal_gate,
                        m3_calibrator=m3_cal_gate,
                    )

                    val_res_gate = evaluate_predictions(
                        f"VAL (HYBRID EXPERT GATE tl={tl:.2f},th={th:.2f},C={c_val:.2f},gc={gate_conf:.2f},c={cal_method})",
                        y_val,
                        val_pred_gate,
                        val_score_gate,
                        print_report=False,
                        route=val_route_gate,
                        threshold_note="variant=expert_gate_m1_m3_v1; tuning=id_val_only",
                    )

                    j_score = (
                        OBJ_W_TPR1 * val_res_gate.tpr_at_1pct_fpr
                        + OBJ_W_TPR5 * val_res_gate.tpr_at_5pct_fpr
                        + OBJ_W_MACRO * val_res_gate.macro_f1
                    )
                    coverage = float(val_res_gate.semantic_coverage) if val_res_gate.semantic_coverage is not None else np.nan
                    cov_term = coverage if np.isfinite(coverage) else 0.0
                    j_score = j_score + OBJ_W_COVERAGE * cov_term
                    pass_macro = bool(val_res_gate.macro_f1 >= (SEM_VAL_MACRO_F1 - 0.02))
                    pass_cov = bool(np.isfinite(coverage) and (MIN_SEM_COVERAGE <= coverage <= MAX_SEM_COVERAGE))

                    rows_gate.append(
                        {
                            "tau_low": float(tl),
                            "tau_high": float(th),
                            "gate_c": float(c_val),
                            "gate_confidence": float(gate_conf),
                            "calibration_method": cal_method,
                            "val_macro_f1": float(val_res_gate.macro_f1),
                            "val_tpr_at_1pct_fpr": float(val_res_gate.tpr_at_1pct_fpr),
                            "val_tpr_at_5pct_fpr": float(val_res_gate.tpr_at_5pct_fpr),
                            "semantic_coverage": coverage,
                            "objective_j": float(j_score),
                            "n_uncertain": int(idx_val_uncertain.size),
                            "is_feasible": bool(pass_macro and pass_cov),
                        }
                    )

if len(rows_gate) == 0:
    raise ValueError("No feasible candidates for expert gate fusion.")

tune_gate = pd.DataFrame(rows_gate)
feasible_gate = tune_gate[tune_gate["is_feasible"]].copy()
if feasible_gate.empty:
    best_gate = tune_gate.sort_values(by=["objective_j", "val_macro_f1", "semantic_coverage"], ascending=False).iloc[0]
else:
    best_gate = feasible_gate.sort_values(by=["objective_j", "val_macro_f1", "semantic_coverage"], ascending=False).iloc[0]

GATE_TAU_LOW = float(best_gate["tau_low"])
GATE_TAU_HIGH = float(best_gate["tau_high"])
GATE_C = float(best_gate["gate_c"])
GATE_CONFIDENCE = float(best_gate["gate_confidence"])
GATE_CAL_METHOD = str(best_gate["calibration_method"])

sem_cal_gate_best, m3_cal_gate_best = _fit_calibrators(y_val, sem_val_raw, m3_val_raw, method=GATE_CAL_METHOD)
m1_cal_gate_best = _fit_m1_calibrator(y_val, m1_val_raw, method=GATE_CAL_METHOD)

mask_val_uncertain_best = (sem_val_raw > GATE_TAU_LOW) & (sem_val_raw < GATE_TAU_HIGH)
idx_val_uncertain_best = np.where(mask_val_uncertain_best)[0]
if idx_val_uncertain_best.size < 20:
    raise ValueError("Expert gate selected degenerate uncertain region (<20 samples).")

y_val_u_best = y_val[idx_val_uncertain_best]
if np.unique(y_val_u_best).size < 2:
    raise ValueError("Expert gate selected uncertain region with a single class.")

p_sem_val_cal_best = np.clip(sem_cal_gate_best(sem_val_raw), 0.0, 1.0)
p_m1_val_cal_best = np.clip(m1_cal_gate_best(m1_val_raw), 0.0, 1.0)
p_m3_val_cal_best = np.clip(m3_cal_gate_best(m3_val_raw), 0.0, 1.0)

sem_pred_u_best = (sem_val_raw[idx_val_uncertain_best] >= SEM_T_STAR).astype(int)
m1_pred_u_best = (m1_val_raw[idx_val_uncertain_best] >= M1_T_STAR).astype(int)
m3_pred_u_best = (m3_val_raw[idx_val_uncertain_best] >= M3_T_STAR).astype(int)

y_gate_u_best = build_expert_gate_targets(
    y_val_u_best,
    sem_pred_u_best,
    m1_pred_u_best,
    m3_pred_u_best,
    p_sem_val_cal_best[idx_val_uncertain_best],
    p_m1_val_cal_best[idx_val_uncertain_best],
    p_m3_val_cal_best[idx_val_uncertain_best],
)
if np.unique(y_gate_u_best).size < 2:
    raise ValueError("Expert gate best candidate produced single-class targets.")

ibvs_val_u_best = ibvs_val_df.iloc[idx_val_uncertain_best].reset_index(drop=True)
X_gate_val_u_best, _ = build_expert_gate_features(
    p_sem_val_cal_best[idx_val_uncertain_best],
    p_m1_val_cal_best[idx_val_uncertain_best],
    p_m3_val_cal_best[idx_val_uncertain_best],
    ibvs_val_u_best,
)

expert_gate_model_best = LogisticRegression(
    max_iter=3000,
    C=GATE_C,
    class_weight="balanced",
    solver="lbfgs",
    multi_class="multinomial",
    random_state=RANDOM_SEED,
)
expert_gate_model_best.fit(X_gate_val_u_best, y_gate_u_best)

expert_gate_note = (
    f"variant=expert_gate_m1_m3_v1; tau=({GATE_TAU_LOW:.2f},{GATE_TAU_HIGH:.2f}); "
    f"gate_C={GATE_C:.2f}; gate_confidence={GATE_CONFIDENCE:.2f}; calibration={GATE_CAL_METHOD}; "
    f"m1_t*={M1_T_STAR:.3f}; m3_t*={M3_T_STAR:.3f}; sem_t*={SEM_T_STAR:.3f}; tuning=id_val_only"
)

val_pred_gate, val_score_gate, val_route_gate, _, _, _, _ = expert_gate_predict(
    X_sem_val,
    X_lex_val,
    X_m3_val,
    tau_low=GATE_TAU_LOW,
    tau_high=GATE_TAU_HIGH,
    gate_model=expert_gate_model_best,
    gate_confidence=GATE_CONFIDENCE,
    ibvs_df=ibvs_val_df,
    sem_threshold=SEM_T_STAR,
    m1_threshold=M1_T_STAR,
    m3_threshold=M3_T_STAR,
    sem_calibrator=sem_cal_gate_best,
    m1_calibrator=m1_cal_gate_best,
    m3_calibrator=m3_cal_gate_best,
)
test_pred_gate, test_score_gate, test_route_gate, _, _, _, _ = expert_gate_predict(
    X_sem_test,
    X_lex_test,
    X_m3_test,
    tau_low=GATE_TAU_LOW,
    tau_high=GATE_TAU_HIGH,
    gate_model=expert_gate_model_best,
    gate_confidence=GATE_CONFIDENCE,
    ibvs_df=ibvs_test_df,
    sem_threshold=SEM_T_STAR,
    m1_threshold=M1_T_STAR,
    m3_threshold=M3_T_STAR,
    sem_calibrator=sem_cal_gate_best,
    m1_calibrator=m1_cal_gate_best,
    m3_calibrator=m3_cal_gate_best,
)
ood_pred_gate, ood_score_gate, ood_route_gate, _, _, _, _ = expert_gate_predict(
    X_sem_ood,
    X_lex_ood,
    X_m3_ood,
    tau_low=GATE_TAU_LOW,
    tau_high=GATE_TAU_HIGH,
    gate_model=expert_gate_model_best,
    gate_confidence=GATE_CONFIDENCE,
    ibvs_df=ibvs_ood_df,
    sem_threshold=SEM_T_STAR,
    m1_threshold=M1_T_STAR,
    m3_threshold=M3_T_STAR,
    sem_calibrator=sem_cal_gate_best,
    m1_calibrator=m1_cal_gate_best,
    m3_calibrator=m3_cal_gate_best,
)

val_res_gate = evaluate_predictions("VAL (HYBRID)", y_val, val_pred_gate, val_score_gate, route=val_route_gate, threshold_note=expert_gate_note)
test_res_gate = evaluate_predictions("TEST (HYBRID)", y_test, test_pred_gate, test_score_gate, route=test_route_gate, threshold_note=expert_gate_note)
ood_res_gate = evaluate_predictions("OOD (HYBRID)", y_ood, ood_pred_gate, ood_score_gate, route=ood_route_gate, threshold_note=expert_gate_note)

hybrid_expert_gate_df = results_to_dataframe("HYBRID_SEM_EXPERT_GATE_M1_M3", [val_res_gate, test_res_gate, ood_res_gate])
hybrid_expert_gate_df["eval_track"] = "deployment_threshold"
hybrid_expert_gate_df["hybrid_variant"] = "expert_gate_m1_m3_v1"
hybrid_expert_gate_df["tau_low"] = float(GATE_TAU_LOW)
hybrid_expert_gate_df["tau_high"] = float(GATE_TAU_HIGH)
hybrid_expert_gate_df["score_definition"] = "semantic outside uncertain; uncertain expert selected by learned multinomial gate over semantic/m1/m3 + ibvs components"
hybrid_expert_gate_df["ibvs_boost_alpha"] = 0.0
hybrid_expert_gate_df["ibvs_high_precision_rule"] = EXPERT_GATE_FEATURE_DEFINITION
hybrid_expert_gate_df["tuning_source"] = "id_val_only"
hybrid_expert_gate_df["fusion_margin"] = float(GATE_CONFIDENCE)
hybrid_expert_gate_df["fusion_alpha"] = float(GATE_C)
hybrid_expert_gate_df["ibvs_gate_definition"] = EXPERT_GATE_FEATURE_DEFINITION
hybrid_expert_gate_df["calibration_method"] = GATE_CAL_METHOD


# Shared M3 threshold metadata
for block in (hybrid_baseline_df, hybrid_boost_df, hybrid_boost_v2_df, hybrid_veto_df, hybrid_meta_df, hybrid_expert_gate_df):
    block["m3_val_t_star"] = float(M3_T_STAR)
    block["m3_threshold_mode"] = M3_T_META.get("selection_mode")
    block["m3_val_macro_f1_best"] = M3_T_META.get("val_macro_f1_best")
    block["m3_val_macro_f1_at_t_star"] = M3_T_META.get("val_macro_f1_at_t_star")
    block["m3_val_fpr_at_t_star"] = M3_T_META.get("val_fpr_at_t_star")
    block["m3_val_tpr_at_t_star"] = M3_T_META.get("val_tpr_at_t_star")
    block["m3_val_fpr_constraint_satisfied"] = M3_T_META.get("val_fpr_constraint_satisfied")

hybrid_metrics = pd.concat([hybrid_baseline_df, hybrid_boost_df, hybrid_boost_v2_df, hybrid_veto_df, hybrid_meta_df, hybrid_expert_gate_df], ignore_index=True)

display(hybrid_metrics)

OUT_DIR = PROJECT_ROOT / "experiments" / "results" / "metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / f"metrics_hybrid_split{SPLIT_TAG}.csv"
if not out_path.name.endswith(f"split{SPLIT_TAG}.csv"):
    raise ValueError("Output filename does not match active SPLIT_TAG.")

hybrid_metrics.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
if not WRITE_MINIMAL_OUTPUTS:
    out_primary_suffix = OUT_DIR / f"metrics_hybrid_split{SPLIT_TAG}__ood-ood_test.csv"
    primary_ood_rows = hybrid_metrics[hybrid_metrics["split"].astype(str).eq("OOD (HYBRID)")].copy()
    if not primary_ood_rows.empty:
        primary_ood_rows["ood_name"] = "ood_test"
        primary_ood_rows.to_csv(out_primary_suffix, index=False)
        print(f"Saved primary OOD-suffixed hybrid metrics: {out_primary_suffix}")
else:
    print("WRITE_MINIMAL_OUTPUTS=True: skipped hybrid OOD-suffixed metrics files.")


# 7b) Bin-level diagnostics (length + lexical complexity) for TEST/OOD across hybrid variants
import re

HYBRID_MODEL_ARTIFACTS = [
    {
        "model": "HYBRID_SEM_GATE_ELSE_M3_IBVS_V2",
        "eval_track": "deployment_threshold",
        "test_pred": np.asarray(test_pred_b, dtype=int),
        "test_proba": np.asarray(test_score_b, dtype=float),
        "ood_pred": np.asarray(ood_pred_b, dtype=int),
        "ood_proba": np.asarray(ood_score_b, dtype=float),
    },
    {
        "model": "HYBRID_SEM_ANCHORED_IBVS_BOOST",
        "eval_track": "deployment_threshold",
        "test_pred": np.asarray(test_pred_u, dtype=int),
        "test_proba": np.asarray(test_score_u, dtype=float),
        "ood_pred": np.asarray(ood_pred_u, dtype=int),
        "ood_proba": np.asarray(ood_score_u, dtype=float),
    },
    {
        "model": "HYBRID_SEM_ANCHORED_IBVS_BOOST_V2",
        "eval_track": "deployment_threshold",
        "test_pred": np.asarray(test_pred_v2, dtype=int),
        "test_proba": np.asarray(test_score_v2, dtype=float),
        "ood_pred": np.asarray(ood_pred_v2, dtype=int),
        "ood_proba": np.asarray(ood_score_v2, dtype=float),
    },
    {
        "model": "HYBRID_SEM_ANCHORED_IBVS_VETO",
        "eval_track": "deployment_threshold",
        "test_pred": np.asarray(test_pred_veto, dtype=int),
        "test_proba": np.asarray(test_score_veto, dtype=float),
        "ood_pred": np.asarray(ood_pred_veto, dtype=int),
        "ood_proba": np.asarray(ood_score_veto, dtype=float),
    },
    {
        "model": "HYBRID_SEM_LEARNED_META_FUSION",
        "eval_track": "deployment_threshold",
        "test_pred": np.asarray(test_pred_meta, dtype=int),
        "test_proba": np.asarray(test_score_meta, dtype=float),
        "ood_pred": np.asarray(ood_pred_meta, dtype=int),
        "ood_proba": np.asarray(ood_score_meta, dtype=float),
    },
    {
        "model": "HYBRID_SEM_EXPERT_GATE_M1_M3",
        "eval_track": "deployment_threshold",
        "test_pred": np.asarray(test_pred_gate, dtype=int),
        "test_proba": np.asarray(test_score_gate, dtype=float),
        "ood_pred": np.asarray(ood_pred_gate, dtype=int),
        "ood_proba": np.asarray(ood_score_gate, dtype=float),
    },
]


# Bin construction reuses shared text/quantile helpers from src.common.notebook_utils.

hybrid_bin_rows = []
stage_specs = [
    ("TEST", df_test["prompt_text"].reset_index(drop=True), np.asarray(y_test, dtype=int), "test"),
    ("OOD",  df_ood["prompt_text"].reset_index(drop=True),  np.asarray(y_ood, dtype=int),  "ood"),
]

for stage_name, prompts_stage, y_stage, key_prefix in stage_specs:
    stats_stage = text_stats(prompts_stage)
    stats_stage["length_bin"] = safe_qcut(stats_stage["token_count"], q=4, prefix="len")
    stats_stage["complexity_bin"] = safe_qcut(stats_stage["lexical_ttr"], q=4, prefix="complex")

    for artifact in HYBRID_MODEL_ARTIFACTS:
        pred_stage = artifact[f"{key_prefix}_pred"]
        score_stage = artifact[f"{key_prefix}_proba"]

        for bin_family in ["length_bin", "complexity_bin"]:
            for bin_label in sorted([b for b in stats_stage[bin_family].dropna().unique()]):
                mask = (stats_stage[bin_family] == bin_label).to_numpy(dtype=bool)
                y_s = y_stage[mask]
                p_s = pred_stage[mask]
                s_s = score_stage[mask]

                n_total = int(mask.sum())
                n_pos = int((y_s == 1).sum())
                n_neg = int((y_s == 0).sum())

                if n_total == 0 or n_pos == 0 or n_neg == 0:
                    continue

                note = (
                    f"bin_eval={bin_family}:{bin_label}; stage={stage_name}; eval_track={artifact['eval_track']}; "
                    f"n_total={n_total}; n_pos={n_pos}; n_neg={n_neg}"
                )

                res = evaluate_predictions(
                    split_name=f"{stage_name}_BIN_{bin_family.upper()}_{str(bin_label).upper()}",
                    y_true=y_s,
                    y_pred=p_s,
                    y_score_for_metrics=s_s,
                    print_report=False,
                    threshold_note=note,
                )

                evasion_rate = float(((y_s == 1) & (p_s == 0)).sum() / max(n_pos, 1))

                df_one = results_to_dataframe(artifact["model"], [res])
                df_one["eval_track"] = artifact["eval_track"]
                df_one["slice_stage"] = stage_name
                df_one["bin_family"] = bin_family
                df_one["bin_label"] = str(bin_label)
                df_one["bin_count"] = n_total
                df_one["bin_positive_count"] = n_pos
                df_one["bin_negative_count"] = n_neg
                df_one["evasion_rate"] = evasion_rate
                df_one["token_count_median"] = float(np.median(stats_stage.loc[mask, "token_count"].astype(float)))
                df_one["token_count_mean"] = float(np.mean(stats_stage.loc[mask, "token_count"].astype(float)))
                df_one["lexical_ttr_median"] = float(np.median(stats_stage.loc[mask, "lexical_ttr"].astype(float)))
                df_one["avg_token_len_median"] = float(np.median(stats_stage.loc[mask, "avg_token_len"].astype(float)))
                df_one["split_tag"] = SPLIT_TAG
                hybrid_bin_rows.append(df_one)

if hybrid_bin_rows:
    hybrid_bins_df = pd.concat(hybrid_bin_rows, ignore_index=True)
    if not WRITE_MINIMAL_OUTPUTS:
        hybrid_bins_path = OUT_DIR / f"metrics_hybrid_bins_split{SPLIT_TAG}.csv"
        hybrid_bins_df.to_csv(hybrid_bins_path, index=False)
        print(f"Saved hybrid bin metrics: {hybrid_bins_path}")
        bins_primary_suffix = OUT_DIR / f"metrics_hybrid_bins_split{SPLIT_TAG}__ood-ood_test.csv"
        bins_primary = hybrid_bins_df.copy()
        if "ood_name" not in bins_primary.columns:
            bins_primary["ood_name"] = "id"
        bins_primary.loc[bins_primary["slice_stage"].astype(str).eq("OOD"), "ood_name"] = "ood_test"
        bins_primary.to_csv(bins_primary_suffix, index=False)
        print(f"Saved primary OOD-suffixed hybrid bin metrics: {bins_primary_suffix}")
    else:
        print("WRITE_MINIMAL_OUTPUTS=True: skipped hybrid bin metrics file outputs.")
    display(hybrid_bins_df.head(16))
else:
    print("No hybrid bin metrics were produced.")


# 7c) Secondary OOD evaluation-only outputs (same tuned params, no OOD tuning)
for ood_name in OOD_SECONDARY_SPLITS:
    y_ood_sec = np.asarray(y_ood_map[ood_name], dtype=int)
    X_sem_ood_sec = X_sem_ood_map[ood_name]
    X_m3_ood_sec = X_m3_ood_map[ood_name]
    ibvs_ood_sec_df = ibvs_ood_df_map[ood_name]

    # Baseline
    ood_pred_b_sec, ood_score_b_sec, ood_route_b_sec, _, _ = hybrid_predict(
        X_sem_ood_sec,
        X_m3_ood_sec,
        tau_low=BASE_TAU_LOW,
        tau_high=BASE_TAU_HIGH,
        m3_threshold=M3_T_STAR,
        ibvs_df=ibvs_ood_sec_df,
        score_variant="baseline",
        gate_mode="legacy",
    )
    ood_res_b_sec = evaluate_predictions("OOD (HYBRID)", y_ood_sec, ood_pred_b_sec, ood_score_b_sec, route=ood_route_b_sec, threshold_note=baseline_note + f"; ood_name={ood_name}")
    hybrid_baseline_sec = results_to_dataframe("HYBRID_SEM_GATE_ELSE_M3_IBVS_V2", [ood_res_b_sec])

    # Boost v1
    ood_pred_u_sec, ood_score_u_sec, ood_route_u_sec, _, _ = hybrid_predict(
        X_sem_ood_sec, X_m3_ood_sec,
        tau_low=BOOST1_TAU_LOW,
        tau_high=BOOST1_TAU_HIGH,
        m3_threshold=M3_T_STAR,
        ibvs_df=ibvs_ood_sec_df,
        score_variant="anchored_ibvs_boost",
        ibvs_boost_alpha=BOOST1_ALPHA,
        fusion_margin=0.0,
        gate_mode="legacy",
    )
    ood_res_u_sec = evaluate_predictions("OOD (HYBRID)", y_ood_sec, ood_pred_u_sec, ood_score_u_sec, route=ood_route_u_sec, threshold_note=boost1_note + f"; ood_name={ood_name}")
    hybrid_boost_sec = results_to_dataframe("HYBRID_SEM_ANCHORED_IBVS_BOOST", [ood_res_u_sec])

    # Boost v2
    ood_pred_v2_sec, ood_score_v2_sec, ood_route_v2_sec, _, _ = hybrid_predict(
        X_sem_ood_sec, X_m3_ood_sec,
        tau_low=BOOST2_TAU_LOW,
        tau_high=BOOST2_TAU_HIGH,
        m3_threshold=M3_T_STAR,
        ibvs_df=ibvs_ood_sec_df,
        score_variant="anchored_ibvs_boost_v2",
        ibvs_boost_alpha=BOOST2_ALPHA,
        fusion_margin=BOOST2_MARGIN,
        gate_mode="high_specific",
        sem_calibrator=sem_cal_v2,
        m3_calibrator=m3_cal_v2,
    )
    ood_res_v2_sec = evaluate_predictions("OOD (HYBRID)", y_ood_sec, ood_pred_v2_sec, ood_score_v2_sec, route=ood_route_v2_sec, threshold_note=boost2_note + f"; ood_name={ood_name}")
    hybrid_boost_v2_sec = results_to_dataframe("HYBRID_SEM_ANCHORED_IBVS_BOOST_V2", [ood_res_v2_sec])

    # Veto
    ood_pred_veto_sec, ood_score_veto_sec, ood_route_veto_sec, _, _ = hybrid_predict(
        X_sem_ood_sec, X_m3_ood_sec,
        tau_low=VETO_TAU_LOW,
        tau_high=VETO_TAU_HIGH,
        m3_threshold=M3_T_STAR,
        ibvs_df=ibvs_ood_sec_df,
        score_variant="anchored_ibvs_veto_v1",
        ibvs_boost_alpha=0.0,
        fusion_margin=VETO_MARGIN,
        gate_mode="high_specific",
        sem_calibrator=sem_cal_veto,
        m3_calibrator=m3_cal_veto,
        semantic_threshold=SEM_T_STAR,
    )
    ood_res_veto_sec = evaluate_predictions("OOD (HYBRID)", y_ood_sec, ood_pred_veto_sec, ood_score_veto_sec, route=ood_route_veto_sec, threshold_note=veto_note + f"; ood_name={ood_name}")
    hybrid_veto_sec = results_to_dataframe("HYBRID_SEM_ANCHORED_IBVS_VETO", [ood_res_veto_sec])

    # Learned-meta
    ood_pred_meta_sec, ood_score_meta_sec, ood_route_meta_sec, _, _, _ = learned_meta_predict(
        X_sem_ood_sec,
        X_m3_ood_sec,
        tau_low=META_TAU_LOW,
        tau_high=META_TAU_HIGH,
        meta_model=meta_model_best,
        meta_threshold=META_THRESHOLD,
        ibvs_df=ibvs_ood_sec_df,
        sem_calibrator=sem_cal_meta,
        m3_calibrator=m3_cal_meta,
    )
    ood_res_meta_sec = evaluate_predictions("OOD (HYBRID)", y_ood_sec, ood_pred_meta_sec, ood_score_meta_sec, route=ood_route_meta_sec, threshold_note=meta_note + f"; ood_name={ood_name}")
    hybrid_meta_sec = results_to_dataframe("HYBRID_SEM_LEARNED_META_FUSION", [ood_res_meta_sec])

    # Expert gate
    ood_pred_gate_sec, ood_score_gate_sec, ood_route_gate_sec, _, _, _, _ = expert_gate_predict(
        X_sem_ood_sec,
        X_lex_ood_map[ood_name],
        X_m3_ood_sec,
        tau_low=GATE_TAU_LOW,
        tau_high=GATE_TAU_HIGH,
        gate_model=expert_gate_model_best,
        gate_confidence=GATE_CONFIDENCE,
        ibvs_df=ibvs_ood_sec_df,
        sem_threshold=SEM_T_STAR,
        m1_threshold=M1_T_STAR,
        m3_threshold=M3_T_STAR,
        sem_calibrator=sem_cal_gate_best,
        m1_calibrator=m1_cal_gate_best,
        m3_calibrator=m3_cal_gate_best,
    )
    ood_res_gate_sec = evaluate_predictions("OOD (HYBRID)", y_ood_sec, ood_pred_gate_sec, ood_score_gate_sec, route=ood_route_gate_sec, threshold_note=expert_gate_note + f"; ood_name={ood_name}")
    hybrid_gate_sec = results_to_dataframe("HYBRID_SEM_EXPERT_GATE_M1_M3", [ood_res_gate_sec])

    hybrid_metrics_sec = pd.concat(
        [hybrid_baseline_sec, hybrid_boost_sec, hybrid_boost_v2_sec, hybrid_veto_sec, hybrid_meta_sec, hybrid_gate_sec],
        ignore_index=True,
    )
    hybrid_metrics_sec["eval_track"] = "deployment_threshold"
    hybrid_metrics_sec["ood_name"] = ood_name

    # Populate shared metadata columns to keep schema comparable.
    meta_template = {
        "m3_val_t_star": float(M3_T_STAR),
        "m3_threshold_mode": M3_T_META.get("selection_mode"),
        "m3_val_fpr_at_t_star": M3_T_META.get("val_fpr_at_t_star"),
        "m3_val_fpr_constraint_satisfied": M3_T_META.get("val_fpr_constraint_satisfied"),
        "tuning_source": "id_val_only",
    }
    for k, v in meta_template.items():
        hybrid_metrics_sec[k] = v

    # Attach variant-level settings.
    variant_defaults = {
        "HYBRID_SEM_GATE_ELSE_M3_IBVS_V2": dict(hybrid_variant="baseline", tau_low=BASE_TAU_LOW, tau_high=BASE_TAU_HIGH, score_definition="p_sem outside uncertain; p_m3 inside uncertain", ibvs_boost_alpha=0.0, ibvs_high_precision_rule=LEGACY_IBVS_GATE_DEFINITION, fusion_margin=0.0, fusion_alpha=0.0, ibvs_gate_definition=LEGACY_IBVS_GATE_DEFINITION, calibration_method="none"),
        "HYBRID_SEM_ANCHORED_IBVS_BOOST": dict(hybrid_variant="anchored_ibvs_boost", tau_low=BOOST1_TAU_LOW, tau_high=BOOST1_TAU_HIGH, score_definition="p_sem globally; uncertain boosted toward p_m3 where legacy ibvs gate and delta>0", ibvs_boost_alpha=BOOST1_ALPHA, ibvs_high_precision_rule=LEGACY_IBVS_GATE_DEFINITION, fusion_margin=0.0, fusion_alpha=BOOST1_ALPHA, ibvs_gate_definition=LEGACY_IBVS_GATE_DEFINITION, calibration_method="none"),
        "HYBRID_SEM_ANCHORED_IBVS_BOOST_V2": dict(hybrid_variant="anchored_ibvs_boost_v2", tau_low=BOOST2_TAU_LOW, tau_high=BOOST2_TAU_HIGH, score_definition="p_sem calibrated; uncertain boosted toward calibrated p_m3 when gate and delta>margin", ibvs_boost_alpha=BOOST2_ALPHA, ibvs_high_precision_rule=HIGH_SPEC_IBVS_GATE_DEFINITION, fusion_margin=BOOST2_MARGIN, fusion_alpha=BOOST2_ALPHA, ibvs_gate_definition=HIGH_SPEC_IBVS_GATE_DEFINITION, calibration_method=BOOST2_CAL_METHOD),
        "HYBRID_SEM_ANCHORED_IBVS_VETO": dict(hybrid_variant="anchored_ibvs_veto_v1", tau_low=VETO_TAU_LOW, tau_high=VETO_TAU_HIGH, score_definition="p_sem calibrated with high-risk veto to semantic high threshold", ibvs_boost_alpha=0.0, ibvs_high_precision_rule=HIGH_SPEC_IBVS_GATE_DEFINITION, fusion_margin=VETO_MARGIN, fusion_alpha=0.0, ibvs_gate_definition=HIGH_SPEC_IBVS_GATE_DEFINITION, calibration_method=VETO_CAL_METHOD),
        "HYBRID_SEM_LEARNED_META_FUSION": dict(hybrid_variant="learned_meta_fusion", tau_low=META_TAU_LOW, tau_high=META_TAU_HIGH, score_definition="meta logistic over calibrated sem/m3 + ibvs interactions in uncertain band", ibvs_boost_alpha=0.0, ibvs_high_precision_rule=LEARNED_META_FEATURE_DEFINITION, fusion_margin=0.0, fusion_alpha=0.0, ibvs_gate_definition=LEARNED_META_FEATURE_DEFINITION, calibration_method=META_CAL_METHOD),
        "HYBRID_SEM_EXPERT_GATE_M1_M3": dict(hybrid_variant="expert_gate_m1_m3_v1", tau_low=GATE_TAU_LOW, tau_high=GATE_TAU_HIGH, score_definition="learned multinomial gate over semantic/m1/m3 using calibrated scores + ibvs in uncertain band", ibvs_boost_alpha=0.0, ibvs_high_precision_rule=EXPERT_GATE_FEATURE_DEFINITION, fusion_margin=GATE_CONFIDENCE, fusion_alpha=GATE_C, ibvs_gate_definition=EXPERT_GATE_FEATURE_DEFINITION, calibration_method=GATE_CAL_METHOD),
    }

    for model_name, defaults in variant_defaults.items():
        mask = hybrid_metrics_sec["model"].eq(model_name)
        for k, v in defaults.items():
            hybrid_metrics_sec.loc[mask, k] = v

    if not WRITE_MINIMAL_OUTPUTS:
        out_sec = OUT_DIR / f"metrics_hybrid_split{SPLIT_TAG}__ood-{ood_name}.csv"
        hybrid_metrics_sec.to_csv(out_sec, index=False)
        print(f"Saved secondary OOD hybrid metrics ({ood_name}): {out_sec}")

    # Secondary OOD bin metrics
    def _bin_rows_for_ood(artifact_rows, prompts_stage, y_stage, ood_name_local):
        stats_stage = text_stats(prompts_stage)
        stats_stage["length_bin"] = safe_qcut(stats_stage["token_count"], q=4, prefix="len")
        stats_stage["complexity_bin"] = safe_qcut(stats_stage["lexical_ttr"], q=4, prefix="complex")

        rows = []
        for artifact in artifact_rows:
            pred_stage = artifact["ood_pred"]
            score_stage = artifact["ood_proba"]
            for bin_family in ["length_bin", "complexity_bin"]:
                for bin_label in sorted([b for b in stats_stage[bin_family].dropna().unique()]):
                    mask = (stats_stage[bin_family] == bin_label).to_numpy(dtype=bool)
                    y_s = y_stage[mask]
                    p_s = pred_stage[mask]
                    s_s = score_stage[mask]
                    n_total = int(mask.sum())
                    n_pos = int((y_s == 1).sum())
                    n_neg = int((y_s == 0).sum())
                    if n_total == 0 or n_pos == 0 or n_neg == 0:
                        continue

                    note = (
                        f"bin_eval={bin_family}:{bin_label}; stage=OOD; eval_track={artifact['eval_track']}; "
                        f"n_total={n_total}; n_pos={n_pos}; n_neg={n_neg}; ood_name={ood_name_local}"
                    )
                    res = evaluate_predictions(
                        split_name=f"OOD_BIN_{bin_family.upper()}_{str(bin_label).upper()}",
                        y_true=y_s,
                        y_pred=p_s,
                        y_score_for_metrics=s_s,
                        print_report=False,
                        threshold_note=note,
                    )
                    evasion_rate = float(((y_s == 1) & (p_s == 0)).sum() / max(n_pos, 1))
                    df_one = results_to_dataframe(artifact["model"], [res])
                    df_one["eval_track"] = artifact["eval_track"]
                    df_one["slice_stage"] = "OOD"
                    df_one["bin_family"] = bin_family
                    df_one["bin_label"] = str(bin_label)
                    df_one["bin_count"] = n_total
                    df_one["bin_positive_count"] = n_pos
                    df_one["bin_negative_count"] = n_neg
                    df_one["evasion_rate"] = evasion_rate
                    df_one["token_count_median"] = float(np.median(stats_stage.loc[mask, "token_count"].astype(float)))
                    df_one["token_count_mean"] = float(np.mean(stats_stage.loc[mask, "token_count"].astype(float)))
                    df_one["lexical_ttr_median"] = float(np.median(stats_stage.loc[mask, "lexical_ttr"].astype(float)))
                    df_one["avg_token_len_median"] = float(np.median(stats_stage.loc[mask, "avg_token_len"].astype(float)))
                    df_one["split_tag"] = SPLIT_TAG
                    df_one["ood_name"] = ood_name_local
                    rows.append(df_one)
        return rows

    artifact_rows_sec = [
        {"model": "HYBRID_SEM_GATE_ELSE_M3_IBVS_V2", "eval_track": "deployment_threshold", "ood_pred": np.asarray(ood_pred_b_sec, dtype=int), "ood_proba": np.asarray(ood_score_b_sec, dtype=float)},
        {"model": "HYBRID_SEM_ANCHORED_IBVS_BOOST", "eval_track": "deployment_threshold", "ood_pred": np.asarray(ood_pred_u_sec, dtype=int), "ood_proba": np.asarray(ood_score_u_sec, dtype=float)},
        {"model": "HYBRID_SEM_ANCHORED_IBVS_BOOST_V2", "eval_track": "deployment_threshold", "ood_pred": np.asarray(ood_pred_v2_sec, dtype=int), "ood_proba": np.asarray(ood_score_v2_sec, dtype=float)},
        {"model": "HYBRID_SEM_ANCHORED_IBVS_VETO", "eval_track": "deployment_threshold", "ood_pred": np.asarray(ood_pred_veto_sec, dtype=int), "ood_proba": np.asarray(ood_score_veto_sec, dtype=float)},
        {"model": "HYBRID_SEM_LEARNED_META_FUSION", "eval_track": "deployment_threshold", "ood_pred": np.asarray(ood_pred_meta_sec, dtype=int), "ood_proba": np.asarray(ood_score_meta_sec, dtype=float)},
        {"model": "HYBRID_SEM_EXPERT_GATE_M1_M3", "eval_track": "deployment_threshold", "ood_pred": np.asarray(ood_pred_gate_sec, dtype=int), "ood_proba": np.asarray(ood_score_gate_sec, dtype=float)},
    ]
    bin_rows_sec = _bin_rows_for_ood(
        artifact_rows_sec,
        df_ood_map[ood_name]["prompt_text"].reset_index(drop=True),
        np.asarray(y_ood_map[ood_name], dtype=int),
        ood_name,
    )
    if bin_rows_sec and (not WRITE_MINIMAL_OUTPUTS):
        hybrid_bins_sec = pd.concat(bin_rows_sec, ignore_index=True)
        bins_sec_path = OUT_DIR / f"metrics_hybrid_bins_split{SPLIT_TAG}__ood-{ood_name}.csv"
        hybrid_bins_sec.to_csv(bins_sec_path, index=False)
        print(f"Saved secondary OOD hybrid bin metrics ({ood_name}): {bins_sec_path}")



Protocol note: no OOD calibration/tuning is used; all parameter search in this notebook uses ID VAL only.


Semantic VAL macro-F1 reference: 0.8601 at t*=0.407

=== VAL (HYBRID) ===
              precision    recall  f1-score   support

           0      0.698     0.925     0.796        40
           1      0.969     0.853     0.907       109

    accuracy                          0.872       149
   macro avg      0.833     0.889     0.852       149
weighted avg      0.896     0.872     0.877       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[37  3]
 [16 93]]
AUC-PR:  0.9077
ROC-AUC: 0.8897
TPR @ FPR: 1%=0.0092, 5%=0.1651, 10%=0.8807
Routing proportions: {'semantic_high': 0.5436, 'fallback_m3': 0.2349, 'semantic_low': 0.2215}

=== TEST (HYBRID) ===
              precision    recall  f1-score   support

           0      0.667     0.927     0.776        41
           1      0.967     0.824     0.890       108

    accuracy                          0.852       149
   macro avg      0.817     0.875     0.833       149
weighted avg      0.885     0.852     0.858       149

Confusion matrix [ [T


=== VAL (HYBRID) ===
              precision    recall  f1-score   support

           0      0.706     0.900     0.791        40
           1      0.959     0.862     0.908       109

    accuracy                          0.872       149
   macro avg      0.833     0.881     0.850       149
weighted avg      0.891     0.872     0.877       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[36  4]
 [15 94]]
AUC-PR:  0.9822
ROC-AUC: 0.9491
TPR @ FPR: 1%=0.7248, 5%=0.8257, 10%=0.8624
Routing proportions: {'semantic_high': 0.6443, 'semantic_low': 0.2953, 'fallback_m3': 0.0604}

=== TEST (HYBRID) ===
              precision    recall  f1-score   support

           0      0.725     0.902     0.804        41
           1      0.959     0.870     0.913       108

    accuracy                          0.879       149
   macro avg      0.842     0.886     0.858       149
weighted avg      0.895     0.879     0.883       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[37  4]
 [14 94]]
AUC-PR:  0.9817


=== VAL (HYBRID) ===
              precision    recall  f1-score   support

           0      0.706     0.900     0.791        40
           1      0.959     0.862     0.908       109

    accuracy                          0.872       149
   macro avg      0.833     0.881     0.850       149
weighted avg      0.891     0.872     0.877       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[36  4]
 [15 94]]
AUC-PR:  0.9822
ROC-AUC: 0.9491
TPR @ FPR: 1%=0.7248, 5%=0.8257, 10%=0.8624
Routing proportions: {'semantic_high': 0.6443, 'semantic_low': 0.2953, 'fallback_m3': 0.0604}

=== TEST (HYBRID) ===
              precision    recall  f1-score   support

           0      0.725     0.902     0.804        41
           1      0.959     0.870     0.913       108

    accuracy                          0.879       149
   macro avg      0.842     0.886     0.858       149
weighted avg      0.895     0.879     0.883       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[37  4]
 [14 94]]
AUC-PR:  0.9817


=== VAL (HYBRID) ===
              precision    recall  f1-score   support

           0      0.756     0.850     0.800        40
           1      0.942     0.899     0.920       109

    accuracy                          0.886       149
   macro avg      0.849     0.875     0.860       149
weighted avg      0.892     0.886     0.888       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[34  6]
 [11 98]]
AUC-PR:  0.9822
ROC-AUC: 0.9491
TPR @ FPR: 1%=0.7248, 5%=0.8257, 10%=0.8624
Routing proportions: {'semantic_high': 0.6443, 'semantic_low': 0.1879, 'semantic_uncertain': 0.1678}

=== TEST (HYBRID) ===
              precision    recall  f1-score   support

           0      0.800     0.878     0.837        41
           1      0.952     0.917     0.934       108

    accuracy                          0.906       149
   macro avg      0.876     0.897     0.886       149
weighted avg      0.910     0.906     0.907       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[36  5]
 [ 9 99]]
AUC-PR: 

...

,model,split,acc,macro_f1,auc_pr,roc_auc,tpr_at_1pct_fpr,tpr_at_5pct_fpr,tpr_at_10pct_fpr,semantic_coverage,...,bin_label,bin_count,bin_positive_count,bin_negative_count,evasion_rate,token_count_median,token_count_mean,lexical_ttr_median,avg_token_len_median,split_tag
0,HYBRID_SEM_GATE_ELSE_M3_IBVS_V2,TEST_BIN_LENGTH_BIN_LEN_Q1,0.842105,0.838068,0.930526,0.875346,0.842105,0.842105,0.842105,None,...,len_q1,38,19,19,0.315789,8.5,7.368421,1.000000,5.111111,C
1,HYBRID_SEM_GATE_ELSE_M3_IBVS_V2,TEST_BIN_LENGTH_BIN_LEN_Q2,0.972973,0.953342,0.998045,0.989247,0.967742,0.967742,0.967742,None,...,len_q2,37,31,6,0.032258,11.0,10.918919,1.000000,5.100000,C
2,HYBRID_SEM_GATE_ELSE_M3_IBVS_V2,TEST_BIN_LENGTH_BIN_LEN_Q3,0.918919,0.839827,0.979816,0.887500,0.562500,0.562500,0.562500,None,...,len_q3,37,32,5,0.062500,14.0,13.783784,0.928571,4.666667,C
3,HYBRID_SEM_GATE_ELSE_M3_IBVS_V2,TEST_BIN_LENGTH_BIN_LEN_Q4,0.675676,0.663636,0.849878,0.783217,0.038462,0.038462,0.346154,None,...,len_q4,37,26,11,0.384615,17.0,25.027027,0.916667,4.750000,C
4,HYBRID_SEM_GATE_ELSE_M3_IBVS_V2,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q1,0.842105,0.817308,0.856670,0.789286,0.035714,0.035714,0.178571,None,...,complex_q1,38,28,10,0.178571,15.0,21.210526,0.888889,4.700000,C
5,HYBRID_SEM_GATE_ELSE_M3_IBVS_V2,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q2,0.891892,0.840517,0.986249,0.942857,0.766667,0.766667,0.766667,None,...,complex_q2,37,30,7,0.100000,14.0,14.864865,0.928571,4.823529,C
6,HYBRID_SEM_GATE_ELSE_M3_IBVS_V2,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q3,0.918919,0.678261,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,complex_q3,37,36,1,0.083333,11.0,11.594595,1.000000,5.100000,C
7,HYBRID_SEM_GATE_ELSE_M3_IBVS_V2,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q4,0.756757,0.700809,0.744577,0.745342,0.214286,0.571429,0.642857,None,...,complex_q4,37,14,23,0.571429,9.0,9.054054,1.000000,5.375000,C
8,HYBRID_SEM_ANCHORED_IBVS_BOOST,TEST_BIN_LENGTH_BIN_LEN_Q1,0.868421,0.866103,0.997368,0.997230,0.947368,0.947368,1.000000,None,...,len_q1,38,19,19,0.263158,8.5,7.368421,1.000000,5.111111,C
9,HYBRID_SEM_ANCHORED_IBVS_BOOST,TEST_BIN_LENGTH_BIN_LEN_Q2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,len_q2,37,31,6,0.000000,11.0,10.918919,1.000000,5.100000,C



=== OOD (HYBRID) ===
              precision    recall  f1-score   support

           0      0.687     0.737     0.711       339
           1      0.717     0.664     0.689       339

    accuracy                          0.701       678
   macro avg      0.702     0.701     0.700       678
weighted avg      0.702     0.701     0.700       678

Confusion matrix [ [TN FP] [FN TP] ]:
 [[250  89]
 [114 225]]
AUC-PR:  0.6830
ROC-AUC: 0.7548
TPR @ FPR: 1%=0.0147, 5%=0.0944, 10%=0.1711
Routing proportions: {'fallback_m3': 0.6593, 'semantic_low': 0.1814, 'semantic_high': 0.1593}

=== OOD (HYBRID) ===
              precision    recall  f1-score   support

           0      0.667     0.655     0.661       339
           1      0.661     0.673     0.667       339

    accuracy                          0.664       678
   macro avg      0.664     0.664     0.664       678
weighted avg      0.664     0.664     0.664       678

Confusion matrix [ [TN FP] [FN TP] ]:
 [[222 117]
 [111 228]]
AUC-PR: 

Saved secondary OOD hybrid bin metrics (ood_test_injection): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_hybrid_bins_splitC__ood-ood_test_injection.csv

=== OOD (HYBRID) ===
              precision    recall  f1-score   support

           0      0.699     0.674     0.686      1993
           1      0.685     0.710     0.698      1993

    accuracy                          0.692      3986
   macro avg      0.692     0.692     0.692      3986
weighted avg      0.692     0.692     0.692      3986

Confusion matrix [ [TN FP] [FN TP] ]:
 [[1343  650]
 [ 577 1416]]
AUC-PR:  0.6226
ROC-AUC: 0.6918
TPR @ FPR: 1%=0.0151, 5%=0.0838, 10%=0.1791
Routing proportions: {'fallback_m3': 0.6249, 'semantic_low': 0.2142, 'semantic_high': 0.1608}

=== OOD (HYBRID) ===
              precision    recall  f1-score   support

           0      0.744     0.721     0.732      1993
           1      0.729     0.752     0.740      1993

    accuracy         


=== OOD (HYBRID) ===
              precision    recall  f1-score   support

           0      0.789     0.627     0.699      1993
           1      0.691     0.832     0.755      1993

    accuracy                          0.730      3986
   macro avg      0.740     0.730     0.727      3986
weighted avg      0.740     0.730     0.727      3986

Confusion matrix [ [TN FP] [FN TP] ]:
 [[1250  743]
 [ 334 1659]]
AUC-PR:  0.7390
ROC-AUC: 0.7831
TPR @ FPR: 1%=0.0000, 5%=0.1957, 10%=0.3307
Routing proportions: {'fallback_meta': 0.5519, 'semantic_low': 0.35, 'semantic_high': 0.0981}

=== OOD (HYBRID) ===
              precision    recall  f1-score   support

           0      0.753     0.650     0.698      1993
           1      0.692     0.787     0.737      1993

    accuracy                          0.719      3986
   macro avg      0.723     0.719     0.717      3986
weighted avg      0.723     0.719     0.717      3986

Confusion matrix [ [TN FP] [FN TP] ]:
 [[1295  698]
 [ 424 1569]]


Saved secondary OOD hybrid bin metrics (ood_test_injection_standard): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_hybrid_bins_splitC__ood-ood_test_injection_standard.csv


In [11]:
# Cell Purpose: Load input datasets/configuration needed for the next processing stage.
# 8) Gate checks for strong hybrid claim vs semantic baseline

METRICS_DIR = PROJECT_ROOT / "experiments" / "results" / "metrics"
semantic_path = METRICS_DIR / f"metrics_semantic_split{SPLIT_TAG}.csv"

if semantic_path.exists():
    semantic_df = pd.read_csv(semantic_path)
    sem_ood = semantic_df[semantic_df["split"] == "OOD"].copy()
    if sem_ood.empty:
        raise ValueError(f"Semantic metrics found but no OOD row in {semantic_path}")
    sem_ref = sem_ood.iloc[0]
    sem_ref_source = f"csv:{semantic_path.name}"
else:
    sem_ood_raw = sem_clf.predict_proba(X_sem_ood)[:, 1]
    sem_ood_pred = (sem_ood_raw >= SEM_T_STAR).astype(int)
    sem_ood_res = evaluate_predictions(
        "OOD (SEM REF)",
        y_ood,
        sem_ood_pred,
        sem_ood_raw,
        print_report=False,
        threshold_note="semantic_ref_in_memory",
    )
    sem_ref = pd.Series(
        {
            "macro_f1": sem_ood_res.macro_f1,
            "tpr_at_1pct_fpr": sem_ood_res.tpr_at_1pct_fpr,
            "tpr_at_5pct_fpr": sem_ood_res.tpr_at_5pct_fpr,
            "tpr_at_10pct_fpr": sem_ood_res.tpr_at_10pct_fpr,
        }
    )
    sem_ref_source = "in_memory"

candidate_ood = hybrid_metrics[
    (hybrid_metrics["model"] == "HYBRID_SEM_LEARNED_META_FUSION")
    & (hybrid_metrics["split"] == "OOD (HYBRID)")
].copy()
if candidate_ood.empty:
    raise ValueError("Missing learned-meta OOD row for claim gating.")
candidate_ood = candidate_ood.iloc[0]

gate_tpr1 = bool(candidate_ood["tpr_at_1pct_fpr"] >= sem_ref["tpr_at_1pct_fpr"])
gate_tpr5 = bool(candidate_ood["tpr_at_5pct_fpr"] >= sem_ref["tpr_at_5pct_fpr"])
gate_macro = bool(candidate_ood["macro_f1"] >= (sem_ref["macro_f1"] - 0.01))

claim_gate = {
    "split_tag": SPLIT_TAG,
    "tuning_source": "id_val_only",
    "semantic_reference_source": sem_ref_source,
    "candidate_model": "HYBRID_SEM_LEARNED_META_FUSION",
    "semantic_ood_macro_f1": float(sem_ref["macro_f1"]),
    "semantic_ood_tpr_at_1pct_fpr": float(sem_ref["tpr_at_1pct_fpr"]),
    "semantic_ood_tpr_at_5pct_fpr": float(sem_ref["tpr_at_5pct_fpr"]),
    "candidate_ood_macro_f1": float(candidate_ood["macro_f1"]),
    "candidate_ood_tpr_at_1pct_fpr": float(candidate_ood["tpr_at_1pct_fpr"]),
    "candidate_ood_tpr_at_5pct_fpr": float(candidate_ood["tpr_at_5pct_fpr"]),
    "must_pass_tpr1": gate_tpr1,
    "must_pass_tpr5": gate_tpr5,
    "stability_macro_f1": gate_macro,
    "strong_claim_allowed_this_split": bool(gate_tpr1 and gate_tpr5 and gate_macro),
}

print("\n=== Hybrid strong-claim gate (current split, ID-only tuning) ===")
print(claim_gate)



=== Hybrid strong-claim gate (current split, ID-only tuning) ===
{'split_tag': 'C', 'tuning_source': 'id_val_only', 'semantic_reference_source': 'csv:metrics_semantic_splitC.csv', 'candidate_model': 'HYBRID_SEM_LEARNED_META_FUSION', 'semantic_ood_macro_f1': 0.7170228445099485, 'semantic_ood_tpr_at_1pct_fpr': 0.203125, 'semantic_ood_tpr_at_5pct_fpr': 0.4401041666666667, 'candidate_ood_macro_f1': 0.746825358670943, 'candidate_ood_tpr_at_1pct_fpr': 0.0, 'candidate_ood_tpr_at_5pct_fpr': 0.4427083333333333, 'must_pass_tpr1': False, 'must_pass_tpr5': True, 'stability_macro_f1': True, 'strong_claim_allowed_this_split': False}


In [12]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# Results Table builder (Repeatability-ready, canonical output only)
# - Loads ablation/semantic/hybrid metrics for Split A/B/C (if present)
# - Enforces hybrid model-family consistency across splits
# - Saves: results_table_v2_repeatability.csv

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()

METRICS_DIR = PROJECT_ROOT / "experiments" / "results" / "metrics"
ARCHIVE_ROOT = PROJECT_ROOT / "experiments" / "results" / "archive"
if not METRICS_DIR.exists():
    ALT = PROJECT_ROOT / "results" / "metrics"
    if ALT.exists():
        METRICS_DIR = ALT

def _latest_named_files(search_roots, pattern):
    latest = {}
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob(pattern):
            current = latest.get(path.name)
            if current is None or path.stat().st_mtime > current.stat().st_mtime:
                latest[path.name] = path
    return latest

print("Using METRICS_DIR:", METRICS_DIR)

SPLIT_TAGS = ["A", "B", "C"]
expected = {
    "ablation": {tag: METRICS_DIR / f"metrics_ablation_split{tag}.csv" for tag in SPLIT_TAGS},
    "semantic": {tag: METRICS_DIR / f"metrics_semantic_split{tag}.csv" for tag in SPLIT_TAGS},
    "transformer": {tag: METRICS_DIR / f"metrics_transformer_split{tag}.csv" for tag in SPLIT_TAGS},
    "transformer_hybrid": {tag: METRICS_DIR / f"metrics_transformer_hybrid_split{tag}.csv" for tag in SPLIT_TAGS},
    "hybrid": {tag: METRICS_DIR / f"metrics_hybrid_split{tag}.csv" for tag in SPLIT_TAGS},
}

EXPECTED_HYBRID_MODELS = {
    "HYBRID_SEM_GATE_ELSE_M3_IBVS_V2",
    "HYBRID_SEM_ANCHORED_IBVS_BOOST",
    "HYBRID_SEM_ANCHORED_IBVS_BOOST_V2",
    "HYBRID_SEM_ANCHORED_IBVS_VETO",
    "HYBRID_SEM_LEARNED_META_FUSION",
    "HYBRID_SEM_EXPERT_GATE_M1_M3",
}
EXPECTED_HYBRID_META_COLS = [
    "eval_track",
    "m3_val_t_star",
    "m3_threshold_mode",
    "m3_val_fpr_at_t_star",
    "m3_val_fpr_constraint_satisfied",
]

STRICT_HYBRID_MODEL_SET_MATCH = False  # Set True once A/B/C have all been rerun with the same hybrid model set.


# Main-table ablation scope for clean dissertation narrative:
# keep baseline ladder + strongest IBVS v2 variants only.
MAIN_ABLATION_MODELS = {
    "M1_TFIDF_ONLY",
    "M2_TFIDF_PLUS_FLAGS",
    "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V1_TOTAL",
    "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL",
    "M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_STRUCTURED",
}

dfs = []
missing = []
hybrid_models_by_split = {}
for exp_name, split_map in expected.items():
    for split_tag, path in split_map.items():
        if path.exists():
            d = pd.read_csv(path)
            if "ood_name" not in d.columns:
                d["ood_name"] = "id"
                if "split" in d.columns:
                    d.loc[d["split"].astype(str).str.startswith("OOD"), "ood_name"] = "ood_test"

            if exp_name == "hybrid":
                models = set(d.get("model", pd.Series(dtype=str)).dropna().astype(str).unique())
                if not models:
                    raise ValueError(f"Hybrid comparability failure in {path.name}: no hybrid models found.")
                if not models.issubset(EXPECTED_HYBRID_MODELS):
                    raise ValueError(
                        f"Hybrid comparability failure in {path.name}: models must be subset of {sorted(EXPECTED_HYBRID_MODELS)}, got {sorted(models)}"
                    )
                hybrid_models_by_split[split_tag] = models
                missing_meta = [c for c in EXPECTED_HYBRID_META_COLS if c not in d.columns]
                if missing_meta:
                    raise ValueError(
                        f"Hybrid schema mismatch in {path.name}; missing columns: {missing_meta}. "
                        "Re-run notebook 06 for this split with the M3 fallback protocol."
                    )

            if "eval_track" not in d.columns:
                d["eval_track"] = "deployment_threshold"
            d["experiment"] = exp_name
            d["split_tag"] = split_tag
            d["source_file"] = path.name
            dfs.append(d)
        else:
            missing.append(str(path))

        # Optional secondary OOD metrics (suffixed files)
        if not WRITE_MINIMAL_OUTPUTS:
            for ood_name in ["ood_test_injection", "ood_test_injection_standard"]:
                ood_filename = f"metrics_{exp_name}_split{split_tag}__ood-{ood_name}.csv"
                latest_ood = _latest_named_files([METRICS_DIR, ARCHIVE_ROOT], ood_filename)
                ood_path = latest_ood.get(ood_filename)
                if ood_path and ood_path.exists():
                    d_ood = pd.read_csv(ood_path)
                    if "ood_name" not in d_ood.columns:
                        d_ood["ood_name"] = ood_name
                    if "eval_track" not in d_ood.columns:
                        d_ood["eval_track"] = "deployment_threshold"
                    d_ood["experiment"] = exp_name
                    d_ood["split_tag"] = split_tag
                    d_ood["source_file"] = ood_path.name
                    dfs.append(d_ood)

if len(hybrid_models_by_split) >= 2:
    unique_sets = {frozenset(v) for v in hybrid_models_by_split.values()}
    if len(unique_sets) != 1:
        mismatch_msg = (
            "Hybrid model sets differ across splits (usually because one split was not rerun yet): "
            f"{hybrid_models_by_split}."
        )
        if STRICT_HYBRID_MODEL_SET_MATCH:
            raise ValueError(f"Hybrid comparability failure across splits: {mismatch_msg}")
        print(f"WARNING: {mismatch_msg} Proceeding with partial comparability.")

if len(dfs) == 0:
    raise FileNotFoundError(
        "No metrics CSVs found. Expected at least one of\n"
        + "\n".join([str(p) for m in expected.values() for p in m.values()])
    )

all_metrics = pd.concat(dfs, ignore_index=True)


# Keep only main ablation models in the canonical repeatability table.
# Raw per-split ablation CSVs still retain all variants for appendix analysis.
if {"experiment", "model"}.issubset(all_metrics.columns):
    before_rows = len(all_metrics)
    ablation_mask = all_metrics["experiment"].eq("ablation")
    keep_mask = (~ablation_mask) | all_metrics["model"].isin(MAIN_ABLATION_MODELS)
    all_metrics = all_metrics.loc[keep_mask].reset_index(drop=True)
    removed_rows = int(before_rows - len(all_metrics))
    if removed_rows > 0:
        print(f"Filtered out {removed_rows} ablation rows (non-main variants) from repeatability table.")

base_cols = [
    "split_tag", "eval_track", "experiment",
    "model", "split", "ood_name",
    "acc", "macro_f1", "auc_pr", "roc_auc",
    "tpr_at_1pct_fpr", "tpr_at_5pct_fpr", "tpr_at_10pct_fpr",
    "semantic_coverage", "defer_rate",
    "threshold_note",
    "val_threshold_t_star", "threshold_selection_mode", "threshold_target_fpr",
    "threshold_macro_f1_tolerance", "threshold_val_fpr_at_t_star",
    "threshold_val_fpr_constraint_satisfied", "threshold_val_fpr_gap_to_target",
    "m3_val_t_star", "m3_threshold_mode", "m3_val_fpr_at_t_star",
    "m3_val_fpr_constraint_satisfied",
    "hybrid_variant", "tau_low", "tau_high", "score_definition",
    "ibvs_boost_alpha", "ibvs_high_precision_rule",
    "tuning_source", "fusion_margin", "fusion_alpha", "ibvs_gate_definition", "calibration_method",
    "source_file",
]
base_present_cols = [c for c in base_cols if c in all_metrics.columns]
extra_cols = [c for c in all_metrics.columns if c not in base_present_cols]
all_metrics = all_metrics[base_present_cols + extra_cols].copy()

metric_cols = [
    "acc", "macro_f1", "auc_pr", "roc_auc",
    "tpr_at_1pct_fpr", "tpr_at_5pct_fpr", "tpr_at_10pct_fpr",
    "semantic_coverage", "defer_rate",
]
for c in metric_cols:
    if c in all_metrics.columns:
        all_metrics[c] = pd.to_numeric(all_metrics[c], errors="coerce").round(4)

# Add repeatability uncertainty summary (mean/std/normal-approx CI95) across split tags,
# focused on OOD rows for claim-facing metrics. Columns are merged back into canonical table.
ood_mask = all_metrics["split"].astype(str).str.startswith("OOD")
agg_metric_cols = ["macro_f1", "tpr_at_1pct_fpr", "tpr_at_5pct_fpr", "tpr_at_10pct_fpr"]
group_cols = ["eval_track", "experiment", "model", "split", "ood_name"]

if ood_mask.any():
    ood_df = all_metrics.loc[ood_mask, group_cols + agg_metric_cols + ["split_tag"]].copy()
    agg_rows = []

    for keys, g in ood_df.groupby(group_cols, dropna=False):
        row = {k: v for k, v in zip(group_cols, keys)}
        n_splits = int(g["split_tag"].nunique())
        row["repeatability_n_splits"] = n_splits

        for m in agg_metric_cols:
            vals = pd.to_numeric(g[m], errors="coerce").dropna().to_numpy(dtype=float)
            if vals.size == 0:
                row[f"{m}_repeat_mean"] = np.nan
                row[f"{m}_repeat_std"] = np.nan
                row[f"{m}_repeat_ci95_low"] = np.nan
                row[f"{m}_repeat_ci95_high"] = np.nan
                continue

            mean = float(np.mean(vals))
            std = float(np.std(vals, ddof=1)) if vals.size > 1 else np.nan
            if vals.size > 1 and np.isfinite(std):
                se = std / np.sqrt(vals.size)
                ci = 1.96 * se
                lo, hi = mean - ci, mean + ci
            else:
                lo, hi = np.nan, np.nan

            row[f"{m}_repeat_mean"] = mean
            row[f"{m}_repeat_std"] = std
            row[f"{m}_repeat_ci95_low"] = lo
            row[f"{m}_repeat_ci95_high"] = hi

        agg_rows.append(row)

    agg_df = pd.DataFrame(agg_rows)
    all_metrics = all_metrics.merge(agg_df, on=group_cols, how="left")

repeat_cols = [c for c in all_metrics.columns if "_repeat_" in c or c == "repeatability_n_splits"]
for c in repeat_cols:
    all_metrics[c] = pd.to_numeric(all_metrics[c], errors="coerce").round(4)


def split_key(s: str) -> int:
    s = str(s)
    if "VAL" in s:
        return 0
    if "TEST" in s:
        return 1
    if "OOD" in s:
        return 2
    return 99

all_metrics["_split_key"] = all_metrics["split"].apply(split_key)
all_metrics = (
    all_metrics
    .sort_values(by=["split_tag", "eval_track", "experiment", "model", "_split_key"])
    .drop(columns=["_split_key"])
    .reset_index(drop=True)
)

loaded_files = [d["source_file"].iloc[0] for d in dfs]
print("Loaded metrics from:")
for f in sorted(set(loaded_files)):
    print(" -", f)

print("\nMissing files (not an error if you haven't run them yet):")
for m in missing:
    print(" -", m)

display(all_metrics)

out_v2 = METRICS_DIR / "results_table_v2_repeatability.csv"
all_metrics.to_csv(out_v2, index=False)
print(f"Saved: {out_v2}")




Using METRICS_DIR: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics
Filtered out 60 ablation rows (non-main variants) from repeatability table.
Loaded metrics from:
 - metrics_ablation_splitA.csv
 - metrics_ablation_splitA__ood-ood_test_injection.csv
 - metrics_ablation_splitA__ood-ood_test_injection_standard.csv
 - metrics_ablation_splitB.csv
 - metrics_ablation_splitB__ood-ood_test_injection.csv
 - metrics_ablation_splitB__ood-ood_test_injection_standard.csv
 - metrics_ablation_splitC.csv
 - metrics_ablation_splitC__ood-ood_test_injection.csv
 - metrics_ablation_splitC__ood-ood_test_injection_standard.csv
 - metrics_hybrid_splitA.csv
 - metrics_hybrid_splitA__ood-ood_test_injection.csv
 - metrics_hybrid_splitA__ood-ood_test_injection_standard.csv
 - metrics_hybrid_splitB.csv
 - metrics_hybrid_splitB__ood-ood_test_injection.csv
 - metrics_hybrid_splitB__ood-ood_test_injection_standard.csv
 - metrics_hybrid_splitC.csv
 - metrics_hybrid_splitC_

,split_tag,eval_track,experiment,model,split,ood_name,acc,macro_f1,auc_pr,roc_auc,...,tpr_at_1pct_fpr_repeat_ci95_low,tpr_at_1pct_fpr_repeat_ci95_high,tpr_at_5pct_fpr_repeat_mean,tpr_at_5pct_fpr_repeat_std,tpr_at_5pct_fpr_repeat_ci95_low,tpr_at_5pct_fpr_repeat_ci95_high,tpr_at_10pct_fpr_repeat_mean,tpr_at_10pct_fpr_repeat_std,tpr_at_10pct_fpr_repeat_ci95_low,tpr_at_10pct_fpr_repeat_ci95_high
0,A,deployment_threshold,ablation,M1_TFIDF_ONLY,VAL,id,0.6577,0.6501,0.9674,0.9183,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A,deployment_threshold,ablation,M1_TFIDF_ONLY,TEST,id,0.6443,0.6348,0.9421,0.8792,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A,deployment_threshold,ablation,M1_TFIDF_ONLY,OOD,ood_test,0.5807,0.5028,0.7249,0.6816,...,0.0626,0.1918,0.3257,0.1806,0.2077,0.4437,0.4768,0.2174,0.3348,0.6188
3,A,deployment_threshold,ablation,M1_TFIDF_ONLY,OOD,ood_test_injection,0.8643,0.8643,0.9282,0.9379,...,0.0626,0.1918,0.3257,0.1806,0.2077,0.4437,0.4768,0.2174,0.3348,0.6188
4,A,deployment_threshold,ablation,M1_TFIDF_ONLY,OOD,ood_test_injection_standard,0.6889,0.6860,0.6895,0.7409,...,0.0626,0.1918,0.3257,0.1806,0.2077,0.4437,0.4768,0.2174,0.3348,0.6188
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250,C,ranking,ablation,M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL,VAL,id,0.8993,0.8639,0.9488,0.9135,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
251,C,ranking,ablation,M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL,TEST,id,0.8859,0.8458,0.9703,0.9402,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
252,C,ranking,ablation,M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL,OOD,ood_test,0.6419,0.6305,0.7479,0.7323,...,0.0159,0.0615,0.1527,0.1158,0.0771,0.2284,0.2459,0.1285,0.1619,0.3299
253,C,ranking,ablation,M3_TFIDF_PLUS_FLAGS_PLUS_IBVS_V2_TOTAL,OOD,ood_test_injection,0.6593,0.6145,0.7171,0.8119,...,0.0159,0.0615,0.1527,0.1158,0.0771,0.2284,0.2459,0.1285,0.1619,0.3299


Saved: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/results_table_v2_repeatability.csv


In [13]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# Difficulty Slice Table builder (canonical cross-model bin diagnostics)
# - Loads ablation/semantic/hybrid bin metrics for Split A/B/C
# - Saves: results_table_difficulty_bins.csv

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
METRICS_DIR = PROJECT_ROOT / "experiments" / "results" / "metrics"
ARCHIVE_ROOT = PROJECT_ROOT / "experiments" / "results" / "archive"
if not METRICS_DIR.exists():
    ALT = PROJECT_ROOT / "results" / "metrics"
    if ALT.exists():
        METRICS_DIR = ALT

def _latest_named_files(search_roots, pattern):
    latest = {}
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob(pattern):
            current = latest.get(path.name)
            if current is None or path.stat().st_mtime > current.stat().st_mtime:
                latest[path.name] = path
    return latest

SPLIT_TAGS = ["A", "B", "C"]
expected_bins = {
    "ablation": {tag: METRICS_DIR / f"metrics_ablation_bins_split{tag}.csv" for tag in SPLIT_TAGS},
    "semantic": {tag: METRICS_DIR / f"metrics_semantic_bins_split{tag}.csv" for tag in SPLIT_TAGS},
    "transformer": {tag: METRICS_DIR / f"metrics_transformer_bins_split{tag}.csv" for tag in SPLIT_TAGS},
    "transformer_hybrid": {tag: METRICS_DIR / f"metrics_transformer_hybrid_bins_split{tag}.csv" for tag in SPLIT_TAGS},
    "hybrid": {tag: METRICS_DIR / f"metrics_hybrid_bins_split{tag}.csv" for tag in SPLIT_TAGS},
}

bin_dfs = []
missing = []
for exp_name, split_map in expected_bins.items():
    for split_tag, path in split_map.items():
        latest_primary = _latest_named_files([METRICS_DIR, ARCHIVE_ROOT], path.name)
        resolved_path = latest_primary.get(path.name)
        if resolved_path and resolved_path.exists():
            d = pd.read_csv(resolved_path)
            d["experiment"] = exp_name
            d["split_tag"] = split_tag
            d["source_file"] = resolved_path.name
            bin_dfs.append(d)
        else:
            missing.append(str(path))
        if not WRITE_MINIMAL_OUTPUTS:
            for ood_name in ["ood_test_injection", "ood_test_injection_standard"]:
                sec_filename = f"metrics_{exp_name}_bins_split{split_tag}__ood-{ood_name}.csv"
                latest_secondary = _latest_named_files([METRICS_DIR, ARCHIVE_ROOT], sec_filename)
                sec_path = latest_secondary.get(sec_filename)
                if sec_path and sec_path.exists():
                    d_sec = pd.read_csv(sec_path)
                    if "ood_name" not in d_sec.columns:
                        d_sec["ood_name"] = ood_name
                    d_sec["experiment"] = exp_name
                    d_sec["split_tag"] = split_tag
                    d_sec["source_file"] = sec_path.name
                    bin_dfs.append(d_sec)

if bin_dfs:
    all_bins = pd.concat(bin_dfs, ignore_index=True)

    # keep key presentation columns first
    base_cols = [
        "split_tag", "experiment", "model", "eval_track", "slice_stage", "ood_name",
        "bin_family", "bin_label", "bin_count", "bin_positive_count", "bin_negative_count",
        "acc", "macro_f1", "auc_pr", "roc_auc",
        "tpr_at_1pct_fpr", "tpr_at_5pct_fpr", "tpr_at_10pct_fpr",
        "evasion_rate", "token_count_median", "token_count_mean",
        "lexical_ttr_median", "avg_token_len_median", "source_file", "threshold_note",
    ]
    cols = [c for c in base_cols if c in all_bins.columns] + [c for c in all_bins.columns if c not in base_cols]
    all_bins = all_bins[cols].copy()

    num_cols = [
        "acc", "macro_f1", "auc_pr", "roc_auc",
        "tpr_at_1pct_fpr", "tpr_at_5pct_fpr", "tpr_at_10pct_fpr",
        "evasion_rate", "token_count_median", "token_count_mean",
        "lexical_ttr_median", "avg_token_len_median",
    ]
    for c in num_cols:
        if c in all_bins.columns:
            all_bins[c] = pd.to_numeric(all_bins[c], errors="coerce").round(4)

    all_bins = all_bins.sort_values(
        by=["split_tag", "slice_stage", "bin_family", "bin_label", "experiment", "model", "eval_track"]
    ).reset_index(drop=True)

    out_bins = METRICS_DIR / "results_table_difficulty_bins.csv"
    all_bins.to_csv(out_bins, index=False)
    print(f"Saved: {out_bins}")
    display(all_bins.head(24))
else:
    print("No bin metrics files found yet; skipping difficulty table build.")

if missing:
    print("Missing bin files (not an error during partial reruns):")
    for m in missing:
        print(" -", m)


Saved: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/results_table_difficulty_bins.csv


,split_tag,experiment,model,eval_track,slice_stage,ood_name,bin_family,bin_label,bin_count,bin_positive_count,...,evasion_rate,token_count_median,token_count_mean,lexical_ttr_median,avg_token_len_median,source_file,threshold_note,split,semantic_coverage,defer_rate
0,A,ablation,M1_TFIDF_ONLY,deployment_threshold,OOD,NaN,complexity_bin,complex_q1,192,79,...,0.7722,11.5,13.8854,0.8824,4.7042,metrics_ablation_bins_splitA.csv,bin_eval=complexity_bin:complex_q1; stage=OOD;...,OOD_BIN_COMPLEXITY_BIN_COMPLEX_Q1,NaN,NaN
1,A,ablation,M1_TFIDF_ONLY,deployment_threshold,OOD,ood_test_injection,complexity_bin,complex_q1,170,86,...,0.0930,118.0,169.2000,0.4788,5.0368,metrics_ablation_bins_splitA__ood-ood_test_inj...,bin_eval=complexity_bin:complex_q1; stage=OOD;...,OOD_BIN_COMPLEXITY_BIN_COMPLEX_Q1,NaN,NaN
2,A,ablation,M1_TFIDF_ONLY,deployment_threshold,OOD,ood_test_injection_standard,complexity_bin,complex_q1,997,722,...,0.0789,214.0,263.4042,0.5877,4.9834,metrics_ablation_bins_splitA__ood-ood_test_inj...,bin_eval=complexity_bin:complex_q1; stage=OOD;...,OOD_BIN_COMPLEXITY_BIN_COMPLEX_Q1,NaN,NaN
3,A,ablation,M1_TFIDF_ONLY,ranking,OOD,NaN,complexity_bin,complex_q1,192,79,...,0.2278,11.5,13.8854,0.8824,4.7042,metrics_ablation_bins_splitA.csv,bin_eval=complexity_bin:complex_q1; stage=OOD;...,OOD_BIN_COMPLEXITY_BIN_COMPLEX_Q1,NaN,NaN
4,A,ablation,M1_TFIDF_ONLY,ranking,OOD,ood_test_injection,complexity_bin,complex_q1,170,86,...,0.0000,118.0,169.2000,0.4788,5.0368,metrics_ablation_bins_splitA__ood-ood_test_inj...,bin_eval=complexity_bin:complex_q1; stage=OOD;...,OOD_BIN_COMPLEXITY_BIN_COMPLEX_Q1,NaN,NaN
5,A,ablation,M1_TFIDF_ONLY,ranking,OOD,ood_test_injection_standard,complexity_bin,complex_q1,997,722,...,0.0028,214.0,263.4042,0.5877,4.9834,metrics_ablation_bins_splitA__ood-ood_test_inj...,bin_eval=complexity_bin:complex_q1; stage=OOD;...,OOD_BIN_COMPLEXITY_BIN_COMPLEX_Q1,NaN,NaN
6,A,ablation,M2_TFIDF_PLUS_FLAGS,deployment_threshold,OOD,NaN,complexity_bin,complex_q1,192,79,...,0.5570,11.5,13.8854,0.8824,4.7042,metrics_ablation_bins_splitA.csv,bin_eval=complexity_bin:complex_q1; stage=OOD;...,OOD_BIN_COMPLEXITY_BIN_COMPLEX_Q1,NaN,NaN
7,A,ablation,M2_TFIDF_PLUS_FLAGS,deployment_threshold,OOD,ood_test_injection,complexity_bin,complex_q1,170,86,...,0.1395,118.0,169.2000,0.4788,5.0368,metrics_ablation_bins_splitA__ood-ood_test_inj...,bin_eval=complexity_bin:complex_q1; stage=OOD;...,OOD_BIN_COMPLEXITY_BIN_COMPLEX_Q1,NaN,NaN
8,A,ablation,M2_TFIDF_PLUS_FLAGS,deployment_threshold,OOD,ood_test_injection_standard,complexity_bin,complex_q1,997,722,...,0.2091,214.0,263.4042,0.5877,4.9834,metrics_ablation_bins_splitA__ood-ood_test_inj...,bin_eval=complexity_bin:complex_q1; stage=OOD;...,OOD_BIN_COMPLEXITY_BIN_COMPLEX_Q1,NaN,NaN
9,A,ablation,M2_TFIDF_PLUS_FLAGS,ranking,OOD,NaN,complexity_bin,complex_q1,192,79,...,0.1519,11.5,13.8854,0.8824,4.7042,metrics_ablation_bins_splitA.csv,bin_eval=complexity_bin:complex_q1; stage=OOD;...,OOD_BIN_COMPLEXITY_BIN_COMPLEX_Q1,NaN,NaN


<!-- NOTEBOOK_OUTPUT_SUMMARY -->
## 6. Output Summary
1. Writes canonical hybrid metrics for the active split to `experiments/results/metrics/metrics_hybrid_split{tag}.csv`.
2. When full-output mode is enabled, also writes OOD-specific suffixed hybrid metrics and hybrid difficulty-bin diagnostics.
3. Rebuilds the consolidated reporting tables:
   - `experiments/results/metrics/results_table_v2_repeatability.csv`
   - `experiments/results/metrics/results_table_difficulty_bins.csv`
4. Provides the split-level hybrid rows used in the final dissertation comparison narrative.
